## ModelEU27 — investment-goods matching
Powered by [Eleonora Priori](https://eleonorapriori.github.io/website/) and [Pietro Terna](https://terna.to.it/) 


In [1]:
import random
import math


def making924table(seed, rankNum, fileName, simulationFirmsExAnteNumber):
    """
    Assign exact global Firm indices from each of the 924 rows to MPI ranks.

    The population represented by each row is created only once, independently
    of rankNum. Changing the number of MPI ranks changes only ownership of the
    global Firm indices, not how many Firms exist or their random characteristics.
    """

    rowData = []
    with open(fileName, newline='') as csvfile:
        firmReader = csv.reader(csvfile, delimiter=',')
        next(firmReader)  # skip header
        for row in firmReader:
            share = 0.0 if row[4] == '' else float(row[4])
            rowData.append((row[0], share))

    if len(rowData) != 924:
        raise RuntimeError(
            f"Expected 924 firm-feature rows, found {len(rowData)}"
        )

    # Sector 44 represents imputed rents and is generated as exactly one
    # virtual Firm per EU27 country. Keep those 27 Firms on the first row
    # belonging to sector 44; any further sector-44 rows create no Firms.
    imputedRentsRow = next(
        (
            rowNumber
            for rowNumber, (sector, _) in enumerate(rowData)
            if sector == '44'
        ),
        None,
    )

    rowFirmCounts = []
    for rowNumber, (sector, share) in enumerate(rowData):
        if sector == '44':
            firmCount = 27 if rowNumber == imputedRentsRow else 0
        else:
            firmCount = round(
                share * simulationFirmsExAnteNumber
            )
        rowFirmCounts.append(firmCount)

    # Stable global Firm addresses are independent of rankNum. Firms are
    # ordered first by the 924 row and then by their row-local Firm index.
    rowFirmOffsets = [0] * len(rowFirmCounts)
    for rowNumber in range(1, len(rowFirmCounts)):
        rowFirmOffsets[rowNumber] = (
            rowFirmOffsets[rowNumber - 1]
            + rowFirmCounts[rowNumber - 1]
        )

    # rowAssignments[row][rank] is a list of half-open intervals
    # (start, stop) of stable Firm indices within that 924 row.
    rowAssignments = [
        [[] for _ in range(rankNum)]
        for _ in range(924)
    ]
    rankLoad = [0] * rankNum

    # Preserve the previous LPT idea, but split exact integer Firm counts.
    # No block contains more than about 1% of the target population.
    maxFirmsPerBlock = max(
        1,
        math.ceil(0.01 * simulationFirmsExAnteNumber),
    )

    blocks = []
    for rowNumber, firmCount in enumerate(rowFirmCounts):
        if firmCount <= 0:
            continue

        nParts = max(
            1,
            math.ceil(firmCount / maxFirmsPerBlock),
        )

        baseSize, remainder = divmod(
            firmCount,
            nParts,
        )

        start = 0
        for partNumber in range(nParts):
            blockSize = (
                baseSize
                + int(partNumber < remainder)
            )
            stop = start + blockSize

            blocks.append((
                blockSize,
                rowNumber,
                partNumber,
                start,
                stop,
            ))
            start = stop

    # Randomize only equal-size ties reproducibly, then apply
    # Longest Processing Time first (LPT) greedy assignment.
    localRandom = random.Random(seed)
    localRandom.shuffle(blocks)
    blocks.sort(
        key=lambda block: block[0],
        reverse=True,
    )

    for blockSize, rowNumber, partNumber, start, stop in blocks:
        targetRank = min(
            range(rankNum),
            key=lambda r: (rankLoad[r], r),
        )

        rowAssignments[rowNumber][targetRank].append(
            (start, stop)
        )
        rankLoad[targetRank] += blockSize

    return rowAssignments, rankLoad, rowFirmOffsets


In [2]:
# libraries for C compilation
import ctypes
import subprocess
import tempfile
import platform
from pathlib import Path

In [3]:
from mpi4py import MPI
from repast4py import context as ctx
import repast4py 
from repast4py import parameters
from repast4py import schedule
from repast4py import core
from typing import Tuple, List, Dict
import numpy as np
import pandas as pd
import pickle
import random
import csv
import os
import sys
import time
import sectors
from itertools import accumulate

comm = MPI.COMM_WORLD
rank    = comm.Get_rank()
rankNum = comm.Get_size() 

if rankNum > 0: print('rank', rank, 'starting',flush=True)

# create the context to hold the agents and manage cross process
# synchronization
context = ctx.SharedContext(comm)

# Initialize the default schedule runner, HERE to create the t() function,
# returning the tick value
runner = schedule.init_schedule_runner(comm)

#Initializes the repast4py.parameters.params dictionary with the model input parameters.
params = parameters.init_params("modelEU27.yaml", "")

# Suppress the extra diagnostics introduced during MPI optimization.
superTechnicalPrint = not params.get('stop_super_technical_print', False)

# Build an exact global population assignment. The same global Firm indices
# exist for every rankNum; only their owner rank changes.
seed924 = params['seed924'] #special independent seed for the 924 table
rowAssignments924, expectedRankLoad, rowFirmOffsets924 = making924table(
    seed924, rankNum, "ff_with_class_limits.csv", params['howManyFirms']
)

# Random characteristics used at Firm creation depend only on the 924 row and
# the Firm's stable index within that row, never on the MPI owner rank.
firmPopulationSeed = int(
    params.get(
        'firmPopulationSeed',
        params['myRandom.seed'][0],
    )
)

if rank == 0 and superTechnicalPrint:
    print("\nEXPECTED FIRM LOAD BY RANK (before special-case adjustments)")
    for r, load in enumerate(expectedRankLoad):
        print(f"rank {r:3d}: {load}")
    minLoad = min(expectedRankLoad)
    maxLoad = max(expectedRankLoad)
    loadRatio = maxLoad / minLoad if minLoad > 0 else float('inf')
    print(
        f"expected load min {minLoad}, max {maxLoad}, "
        f"max/min ratio {loadRatio:.4f}\n"
    )

#generate random seed
repast4py.random.init(rng_seed=params['myRandom.seed'][rank]) #each rank has a seed
rng = repast4py.random.default_rng # repast seed
rng_np = np.random.default_rng(params['myRandom.seed'][rank]) # numpy seed

#timer T()
startTime=-1
def T():
    global startTime
    if startTime < 0:
        startTime=time.time()
    return time.time() - startTime
T() #launches the timer

#cpuTimer Tc()
startCpuTime=-1
def Tc():
    global startCpuTime
    if startCpuTime < 0:
        startCpuTime=time.process_time()
    return time.process_time() - startCpuTime
Tc() #launches the cpu timer

#read control key from firm-features-generation
with open("control4text_ff.txt", "r") as f:
    control = int(f.read())


rank 0 starting


In [4]:
class AgentProxy:

    __slots__ = ("address",)

    def __init__(self, address: int):
        # Stable global address of the real agent in shared arrays.
        self.address = int(address)


PRODUCTION_TYPE_TO_CODE = {
    "C": 0,
    "I": 1,
    "Int": 2,
}

PRODUCTION_TYPE_FROM_CODE = {
    code: productionType
    for productionType, code in PRODUCTION_TYPE_TO_CODE.items()
}


class Firm(core.Agent):

    TYPE = 0
    sharedOutputWarehouse = None
    sharedOutputWarehouseSnapshot = None
    sharedTargetAddedValue = None
    sharedLostSales = None
    sharedBuyerRefill = None
    sharedMatchedSupplierAddress = None
    sharedCountry = None
    sharedSector = None
    sharedProductionTypeCode = None
    localIntermediateWarehouse = None
    localInvestmentGoodsPurchased = None
    localInvestmentPotentialExports = None
    
    def __init__(self, local_id: int, rank: int, sector: int, labor: int, capital: float, capitalR: float, wage: float,\
                 country: int, recipe: float, countryRecipe: float, laborProductivity: float, avgPlannedMarkup: float,\
                 productionType: str):
                 # orderObservationFrequency: int, 
                 
        super().__init__(id=local_id, type=Firm.TYPE, rank=rank) #uid
    
        self.sector=sector 
        self.labor=labor
        self.capital=capital
        self.capitalR=capitalR
        self.wage=wage
        self.country = country
        #self.investment = investment # serviva per interagire col planner ?
        self.recipe = recipe
        self.countryRecipe = countryRecipe
        self.laborProductivity=laborProductivity
        self.plannedMarkup=avgPlannedMarkup
        self.productionType = productionType
        self.address = -1
        self._targetAddedValue = 0.0
        self.actualAddedValue = 0 
        self.warehouseForPotentialExtraEU27export = 0

        # here we introduce an estimated value of added value in t-1 to generate the intermediate warehouse quantities available at t=0
        self.estimatedAddedValueTminus1 = float(self.labor * self.wage + self.capital * 1000 * self.capitalR) / 12
        self.estimatedAddedValueTminus1 *= self.plannedMarkup
        
        # self.orderObservationFrequency=orderObservationFrequency

    @property
    def intermediateWarehouse(self):
        # Compact local storage: one contiguous float64 matrix per MPI rank.
        # uid[0] is the stable local row assigned when the Firm is created.
        return Firm.localIntermediateWarehouse[self.uid[0]]

    @property
    def investmentGoodsPurchased(self):
        # Monthly investment-goods purchases in the same units as Firm.capital.
        return float(Firm.localInvestmentGoodsPurchased[self.uid[0]])

    @investmentGoodsPurchased.setter
    def investmentGoodsPurchased(self, value):
        Firm.localInvestmentGoodsPurchased[self.uid[0]] = value

    @property
    def warehouseForPotentialInvestmentExtraEU27export(self):
        # Current-month residual output of investment-goods Firms available for
        # potential extra-EU exports. Stored compactly outside the Python object.
        return float(Firm.localInvestmentPotentialExports[self.uid[0]])

    @warehouseForPotentialInvestmentExtraEU27export.setter
    def warehouseForPotentialInvestmentExtraEU27export(self, value):
        Firm.localInvestmentPotentialExports[self.uid[0]] = value

    @property
    def outputWarehouse(self):
        return float(Firm.sharedOutputWarehouse[self.address])

    @outputWarehouse.setter
    def outputWarehouse(self, value):
        Firm.sharedOutputWarehouse[self.address] = value

    @property
    def targetAddedValue(self):
        if self.address < 0 or Firm.sharedTargetAddedValue is None:
            return float(self._targetAddedValue)
        return float(Firm.sharedTargetAddedValue[self.address])

    @targetAddedValue.setter
    def targetAddedValue(self, value):
        if self.address < 0 or Firm.sharedTargetAddedValue is None:
            self._targetAddedValue = float(value)
        else:
            Firm.sharedTargetAddedValue[self.address] = value

    @property
    def lostSales(self):
        # Monthly value of demand addressed to this Firm that could not be
        # fulfilled because its output warehouse was empty or insufficient.
        return float(Firm.sharedLostSales[self.address])

    @lostSales.setter
    def lostSales(self, value):
        Firm.sharedLostSales[self.address] = value

    def save(self):
        # Compact serialization remains available for Repast4Py operations,
        # although remote suppliers are represented by FirmProxy objects.
        return (
            self.uid,
            self.sector,
            self.country,
            self.productionType,
            self.address,
        )


class FirmProxy(AgentProxy):

    __slots__ = ()
    TYPE = Firm.TYPE

    @property
    def country(self):
        return int(
            Firm.sharedCountry[self.address]
        )

    @property
    def sector(self):
        return int(
            Firm.sharedSector[self.address]
        )

    @property
    def productionType(self):
        productionTypeCode = int(
            Firm.sharedProductionTypeCode[self.address]
        )
        return PRODUCTION_TYPE_FROM_CODE[
            productionTypeCode
        ]

    @property
    def outputWarehouse(self):
        return float(
            Firm.sharedOutputWarehouse[self.address]
        )

    @property
    def targetAddedValue(self):
        return float(
            Firm.sharedTargetAddedValue[self.address]
        )

    @property
    def lostSales(self):
        return float(
            Firm.sharedLostSales[self.address]
        )


In [5]:
def statistics(y_true, y_est):

    y_true = np.array(y_true)
    y_est  = np.array(y_est)

    n = y_true.shape[0]
    
    #MAE, Mean Absolute Error
    # $ MAE = \frac{1}{n}\sum_{i=1}^n | yi - \hat{y}_i | $
    mae = abs(y_true - y_est).mean()

    # RMSE, Root Mean Squared Error
    # $ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n ( y_i - \hat{y}_i )^2} $
    rmse = (((y_true - y_est)**2).sum()/n)**0.5

    # MAPE, Mean Absolute Percentage Error
    # $ MAPE = \frac{100}{n}\sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right| $
    safeDiv = np.where(y_true == 0, 0.000001, 0)
    mape = (abs((y_true - y_est) / (y_true+safeDiv)).sum()) * 100 / n

    # R², coefficient of determination
    # $ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $
    if ((y_true - y_true.mean())**2).sum() == 0: r2 = 0
    else: r2 = 1 - (((y_true - y_est)**2).sum()/((y_true - y_true.mean())**2).sum())**0.5

    return (mae,rmse,mape,r2,n)  #,rmse, mape,r2,count)

  
  
$ MAE = \frac{1}{n}\sum_{i=1}^n | yi - \hat{y}_i | $

$ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n ( y_i - \hat{y}_i )^2} $

$ MAPE = \frac{100}{n}\sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right| $

$ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $


In [6]:
from IPython.display import display, HTML
def stop_execution(message):
    display(HTML(f"<pre>{message}</pre>"))
    raise SystemExit(0)

In [7]:
def generateOutput(countryLabel, modelOutput, AFF_FirmNumber, countAFF_Firms, propFactor, AFF_WorkerNumber, countAFF_WorkerNumber,
                   Eu_FirmNumber, countActualExPostFirmNumber, Eu_WorkerNumber, countWorkers, Eu_EmployeeCompensations,
                   countEmployeeCompensations, Eu_GDP_withVAT2022, Eu_AddedValue2022, countCapitalR, countActualAddedValue, markupTentative,
                   Eu_Intermediate2022, intermediateTotal, substitutionRate, substitutionRateL, firmSubstitutions,
                   allfirmSubstitutionsByBuyerSector, simulationFirmsExAnteNumber, allfirmSubstitutionsByVendorSector,
                   intermediateTotalImportsByCountryAndSector, intermediateTotalPotentialExportsByCountryAndSector,
                   investmentTotalImportsByCountryAndSector, investmentTotalPotentialExportsByCountryAndSector):
    
    print("INT IMPORT", round(float(intermediateTotalImportsByCountryAndSector.sum())/1_000_000,2),\
          file=modelOutput, flush = True)
    print([(i,round(float(intermediateTotalImportsByCountryAndSector[i])/ 1_000_000, 2)) for i in range(len(intermediateTotalImportsByCountryAndSector))]\
                                                                            ,file=modelOutput, flush = True)
    print("POTENTIAL EXPORT", round(float(intermediateTotalPotentialExportsByCountryAndSector.sum())/1_000_000,2),\
          file=modelOutput, flush = True)
    print([(i,round(float(intermediateTotalPotentialExportsByCountryAndSector[i])/ 1_000_000, 2)) for i in range(len(intermediateTotalPotentialExportsByCountryAndSector))]\
                                                                            ,file=modelOutput, flush = True)

    print("INV IMPORT", round(float(investmentTotalImportsByCountryAndSector.sum())/1_000_000,2),\
          file=modelOutput, flush = True)
    print([(i,round(float(investmentTotalImportsByCountryAndSector[i])/ 1_000_000, 2)) for i in range(len(investmentTotalImportsByCountryAndSector))]\
                                                                            ,file=modelOutput, flush = True)
    print("INV POTENTIAL EXPORT", round(float(investmentTotalPotentialExportsByCountryAndSector.sum())/1_000_000,2),\
          file=modelOutput, flush = True)
    print([(i,round(float(investmentTotalPotentialExportsByCountryAndSector[i])/ 1_000_000, 2)) for i in range(len(investmentTotalPotentialExportsByCountryAndSector))]\
                                                                            ,file=modelOutput, flush = True)

    # employeeCompensations_CapitalR_markupAugmented = \
    #   int(markupTentative*(countEmployeeCompensations/1_000_000 + countCapitalR/1000))
    capitalR_fullMarkupAugmented = \
       int(countCapitalR/1000 + (markupTentative-1)*(countEmployeeCompensations/1_000_000 + countCapitalR/1000))
    table_data = [
        [' ',          'EU 27', 'simulation', 'simulation to'],
        [' ',          ' ',     ' ',          'EU scale'],
        ['AFF Firm Number', AFF_FirmNumber, countAFF_Firms, int(countAFF_Firms * propFactor)], 
        ['AFF Worker Number', AFF_WorkerNumber, countAFF_WorkerNumber, int(countAFF_WorkerNumber * propFactor)], 
        ['Firm Number', Eu_FirmNumber, countActualExPostFirmNumber, int(countActualExPostFirmNumber * propFactor)],
        ['Worker Number', Eu_WorkerNumber, countWorkers, int(countWorkers * propFactor)],
        ['Empl. Compens.', int(Eu_EmployeeCompensations), int(countEmployeeCompensations/1_000_000), 
                           int(countEmployeeCompensations * propFactor / 1_000_000)],                                                                                                                              
        ['Cap. R & Op. Surplus', int(Eu_AddedValue2022 - Eu_EmployeeCompensations), capitalR_fullMarkupAugmented, 
                                 int(capitalR_fullMarkupAugmented*propFactor)],
        ['GDP curr. p. (VAT)', Eu_GDP_withVAT2022, 'N.B. contains VAT',' '],
        ['Add. Val. basic p.', Eu_AddedValue2022, int(countActualAddedValue/1_000_000), 
                                # countActualAddedValue was employeeCompensations_CapitalR_markupAugmented in the previous version
                               int((countActualAddedValue/1_000_000)*propFactor)],
        ['Int. Prod. basic p.', int(Eu_Intermediate2022), int(intermediateTotal/1000_000),
         int(intermediateTotal * propFactor / 1000000)]
    ]
    
    print("\n", countryLabel, "\n", file=modelOutput, flush=True)
    
    print(f"Markup tentative {100*(markupTentative-1)}%", file=modelOutput, flush=True)
    for row in table_data:
        print("{: >20} {: >20} {: >20} {: >20}".format(*row), file=modelOutput, flush=True)

    print("\n AFF => Agriculture, Forestry and Fishing\n\n", file=modelOutput, flush=True)


In [8]:
def generateGDPOutput(countryLabel, modelOutput, propFactor, countActualAddedValue):
                        #countEmployeeCompensations, countCapitalR, markupTentative):
    
    print(countryLabel, int((countActualAddedValue/1_000_000) * propFactor),
          #int(markupTentative*(countEmployeeCompensations * propFactor/1_000_000 + countCapitalR * propFactor/1000)), \
          file=modelOutput)

In [9]:
def deterministic_firm_uniform(address: int, period: int, seed: int) -> float:
    """Return a reproducible U[0,1) draw independent of MPI ownership."""

    mask = (1 << 64) - 1

    x = (
        int(address)
        + 0x9E3779B97F4A7C15
        + (int(period) + 1) * 0xD1B54A32D192ED03
        + (int(seed) + 1) * 0x94D049BB133111EB
    ) & mask

    x = ((x ^ (x >> 30)) * 0xBF58476D1CE4E5B9) & mask
    x = ((x ^ (x >> 27)) * 0x94D049BB133111EB) & mask
    x = (x ^ (x >> 31)) & mask

    return (x >> 11) / float(1 << 53)


def match_global_buyer_addresses_to_suppliers(
    buyer_addresses,
    suppliers,
    buyer_country: int,
    mult: float,
    rng: np.random.Generator,
    supplier_output_snapshot,
    buyer_refill_snapshot,
    point_mode: str = "systematic",
):
    """
    Match all EU27 buyers of one country to suppliers of one input sector.

    This is a global Two-Tape match: ``buyer_addresses`` contains every Firm
    of the buyer country, independently of MPI ownership. The buyer tape is
    ordered by stable global Firm address, and the supplier list is ordered by
    stable global Firm address before the deterministic permutation is applied.

    Tape A is weighted by current residual refill. Tape B is weighted by the
    common frozen outputWarehouse snapshot, with same-country supplier weights
    multiplied by ``mult``. The result is the matched supplier global address
    for every buyer address; -1 means that no supplier contact is possible.
    """

    buyer_addresses = np.asarray(
        buyer_addresses,
        dtype=np.int64,
    )

    n_buyers = buyer_addresses.size
    n_suppliers = len(suppliers)

    matched_supplier_addresses = np.full(
        n_buyers,
        -1,
        dtype=np.int64,
    )

    if n_buyers == 0:
        return matched_supplier_addresses

    if n_suppliers == 0:
        return matched_supplier_addresses

    if mult <= 0.0:
        raise ValueError("mult must be strictly positive.")

    if not 0 <= buyer_country <= 26:
        raise ValueError("buyer_country must be between 0 and 26.")

    # ------------------------------------------------------------
    # 1. Global buyer tape, ordered by stable Firm address
    # ------------------------------------------------------------

    buyer_weights = np.maximum(
        buyer_refill_snapshot[buyer_addresses],
        0.0,
    )

    if np.any(~np.isfinite(buyer_weights)):
        raise ValueError(
            "Global buyer refill snapshot contains non-finite values."
        )

    total_buyer_weight = buyer_weights.sum()

    if total_buyer_weight <= 0.0:
        return matched_supplier_addresses

    # ------------------------------------------------------------
    # 2. Supplier tape, initially ordered by stable Firm address
    # ------------------------------------------------------------

    supplier_addresses = np.fromiter(
        (supplier.address for supplier in suppliers),
        dtype=np.int64,
        count=n_suppliers,
    )

    supplier_countries = np.fromiter(
        (supplier.country for supplier in suppliers),
        dtype=np.int16,
        count=n_suppliers,
    )

    supplier_weights = np.maximum(
        supplier_output_snapshot[supplier_addresses],
        0.0,
    )

    if np.any(~np.isfinite(supplier_weights)):
        raise ValueError(
            "outputWarehouse snapshot contains non-finite values."
        )

    same_country = (
        supplier_countries == buyer_country
    )

    supplier_weights = supplier_weights.copy()
    supplier_weights[same_country] *= mult

    total_supplier_weight = supplier_weights.sum()

    if total_supplier_weight <= 0.0:
        return matched_supplier_addresses

    # ------------------------------------------------------------
    # 3. One rank-invariant permutation of the supplier tape
    # ------------------------------------------------------------

    permutation = rng.permutation(n_suppliers)

    permuted_supplier_addresses = (
        supplier_addresses[permutation]
    )

    permuted_supplier_weights = (
        supplier_weights[permutation]
    )

    cumulative_supplier_weights = np.cumsum(
        permuted_supplier_weights
    )

    cumulative_supplier_weights[-1] = (
        total_supplier_weight
    )

    # ------------------------------------------------------------
    # 4. Global quantity-weighted buyer positions
    # ------------------------------------------------------------

    positive_buyer_mask = (
        buyer_weights > 0.0
    )

    positive_buyer_weights = (
        buyer_weights[positive_buyer_mask]
    )

    buyer_segment_starts = np.concatenate((
        np.array([0.0], dtype=np.float64),
        np.cumsum(positive_buyer_weights[:-1]),
    ))

    if point_mode == "starts":

        buyer_positions = buyer_segment_starts

    elif point_mode == "systematic":

        # The same fractional offset is used inside every buyer segment.
        offset = rng.random()

        buyer_positions = (
            buyer_segment_starts
            + offset * positive_buyer_weights
        )

    elif point_mode == "stratified":

        buyer_positions = (
            buyer_segment_starts
            + rng.random(len(positive_buyer_weights))
            * positive_buyer_weights
        )

    else:
        raise ValueError(
            "point_mode must be 'starts', "
            "'systematic' or 'stratified'."
        )

    buyer_points = (
        buyer_positions
        * total_supplier_weight
        / total_buyer_weight
    )

    matched_positions = np.searchsorted(
        cumulative_supplier_weights,
        buyer_points,
        side="right",
    ).astype(np.int64)

    matched_supplier_addresses[
        positive_buyer_mask
    ] = permuted_supplier_addresses[
        matched_positions
    ]

    return matched_supplier_addresses


In [10]:
def printDemandOfferRefill(outAddedV, intermediateInputs):

    ##### added value int, C, I, and tot
    if outAddedV:
         totalTargetAddedValueBySectIntProduction = [0] * 64
         totalActualAddedValueBySectIntProduction = [0] * 64
         for aFirm in context.agents(agent_type=0):
             if aFirm.productionType == "Int":
                 totalTargetAddedValueBySectIntProduction[aFirm.sector - 1] += aFirm.targetAddedValue
                 totalActualAddedValueBySectIntProduction[aFirm.sector - 1] += aFirm.actualAddedValue

         print(
         "\nInt. good firms: target and actual added value by sector\n",
         [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
             for i, (target, actual) in enumerate(
                 zip(totalTargetAddedValueBySectIntProduction, totalActualAddedValueBySectIntProduction))])
         t1 = sum(totalTargetAddedValueBySectIntProduction)
         t2 = sum(totalActualAddedValueBySectIntProduction)
         print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))

    
         totalTargetAddedValueBySectCProduction = [0] * 64
         totalActualAddedValueBySectCProduction = [0] * 64
         for aFirm in context.agents(agent_type=0):
             if aFirm.productionType == "C":
                 totalTargetAddedValueBySectCProduction[aFirm.sector - 1] += aFirm.targetAddedValue
                 totalActualAddedValueBySectCProduction[aFirm.sector - 1] += aFirm.actualAddedValue

         print(
         "\nCons. good firms: target and actual added value by sector\n",
         [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
             for i, (target, actual) in enumerate(
                 zip(totalTargetAddedValueBySectCProduction, totalActualAddedValueBySectCProduction))])
         t1 = sum(totalTargetAddedValueBySectCProduction)
         t2 = sum(totalActualAddedValueBySectCProduction)
         print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))

    
         totalTargetAddedValueBySectIProduction = [0] * 64
         totalActualAddedValueBySectIProduction = [0] * 64
         for aFirm in context.agents(agent_type=0):
             if aFirm.productionType == "I":
                 totalTargetAddedValueBySectIProduction[aFirm.sector - 1] += aFirm.targetAddedValue
                 totalActualAddedValueBySectIProduction[aFirm.sector - 1] += aFirm.actualAddedValue

         print(
         "\nInv. good firms: target and actual added value by sector\n",
         [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
             for i, (target, actual) in enumerate(
                 zip(totalTargetAddedValueBySectIProduction, totalActualAddedValueBySectIProduction))])
         t1 = sum(totalTargetAddedValueBySectIProduction)
         t2 = sum(totalActualAddedValueBySectIProduction)
         print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))


         totalTargetAddedValueBySectTProduction = [0] * 64
         totalActualAddedValueBySectTProduction = [0] * 64
         for aFirm in context.agents(agent_type=0):
         
             totalTargetAddedValueBySectTProduction[aFirm.sector - 1] += aFirm.targetAddedValue
             totalActualAddedValueBySectTProduction[aFirm.sector - 1] += aFirm.actualAddedValue

         print(
         "\nAll good firms: target and actual added value by sector\n",
         [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
             for i, (target, actual) in enumerate(
                 zip(totalTargetAddedValueBySectTProduction, totalActualAddedValueBySectTProduction))])
         t1 = sum(totalTargetAddedValueBySectTProduction)
         t2 = sum(totalActualAddedValueBySectTProduction)
         print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))

    ##### demand of intermediate products by firms of each speficic sector, any productionType (Naio cols)
    # calculations here have uniquely a diagnostic goal
    
    totalIntermediateWarehouseRefillByUsingSect = [0] * 64
    for aFirm in context.agents(agent_type=0):
        warehouse = aFirm.intermediateWarehouse
        for i in range(64):
            refill = (
                aFirm.targetAddedValue * intermediateInputs[i][aFirm.sector - 1]
                - warehouse[i]
            )
            totalIntermediateWarehouseRefillByUsingSect[aFirm.sector-1] += refill

    print(
        "\nAll firms refill needs: demand of int. goods by using sector, any productionType (sum Naio col)\n",
        [(i + 1, round(value / 1_000_000, 2))
         for i, value in enumerate(totalIntermediateWarehouseRefillByUsingSect)]
    )
    t1 = sum(totalIntermediateWarehouseRefillByUsingSect)
    print("TOTAL:", round(t1 / 1_000_000, 2))

    ##### demand of intermediate products by all the producing sector from each speficic sector (Naio rows) 

    totalIntermediateWarehouseRefillByProducingSect = [0] * 64
    for aFirm in context.agents(agent_type=0):
        warehouse = aFirm.intermediateWarehouse
        for i in range(64):
            refill = (
                aFirm.targetAddedValue * intermediateInputs[i][aFirm.sector - 1]
                - warehouse[i]
            )
            totalIntermediateWarehouseRefillByProducingSect[i] += refill

    print(
        "\nAll firms refill needs: demand of int. goods from a specific sector by all producing sector (sum Naio row)\n",
        [(i + 1, round(value / 1_000_000, 2))
         for i, value in enumerate(totalIntermediateWarehouseRefillByProducingSect)]
    )
    t1 = sum(totalIntermediateWarehouseRefillByProducingSect)
    print("TOTAL:", round(t1 / 1_000_000, 2))


    
    # offer of int goods 
    totalOutputWarehousesOfIntermediateGoods = [0] * 64
    for aFirm in context.agents(agent_type=0):
        if aFirm.productionType == "Int":
            totalOutputWarehousesOfIntermediateGoods[aFirm.sector - 1] += aFirm.outputWarehouse

    print(
        "\nInt good firms output warehouses by sector\n",
        [(i + 1, round(value / 1_000_000, 2))
         for i, value in enumerate(totalOutputWarehousesOfIntermediateGoods)]
    )
    t2 = sum(totalOutputWarehousesOfIntermediateGoods)
    print("TOTAL:", round(t2 / 1_000_000, 2))


    print(
        "\nDifferences\n",
        [(i + 1, round((a - b) / 1_000_000, 2))
         for i, (a, b) in enumerate(zip(
             totalOutputWarehousesOfIntermediateGoods,
             totalIntermediateWarehouseRefillByProducingSect
         ))]
    )
    print("TOTAL DIFF:", round((t2 - t1) / 1_000_000, 2))

In [11]:
def scalingWorkerDistribution(worker_distribution):
    values = [0] * 1849
    # class 1-9
    tot = 0
    for i in range(9): 
        tot += worker_distribution[i]
    for i in range(9):
        values[i] = worker_distribution[i] / tot
    # class 10-19
    tot = 0 
    for i in range(9, 19):
        tot += worker_distribution[i]
    for i in range(9, 19):
        values[i] = worker_distribution[i] / tot
    # class 20-49
    tot = 0 
    for i in range(19, 49):
        tot += worker_distribution[i]
    for i in range(19, 49):
        values[i] = worker_distribution[i] / tot
    # class 50-249
    tot = 0 
    for i in range(49, 249):
        tot += worker_distribution[i]
    for i in range(49, 249):
        values[i] = worker_distribution[i] / tot
    # class 250 or more (1849)
    tot = 0 
    for i in range(249, 1849):
        tot += worker_distribution[i]
    for i in range(249, 1849):
        values[i] = worker_distribution[i] / tot     
    return values



def buildAtomicLockHelper(sharedComm):

    # This tiny C11 helper performs CPU atomic operations directly on
    # the shared-memory lock array. No MPI operation occurs per purchase.
    helperDir = Path(tempfile.gettempdir()) / "modelEU27_atomic_transaction_v2"
    helperDir.mkdir(parents=True, exist_ok=True)

    sourcePath = helperDir / "atomic_lock.c"
    libraryPath = helperDir / (
        "libatomic_lock.dylib"
        if platform.system() == "Darwin"
        else "libatomic_lock.so"
    )

    source = """
#include <stdint.h>
#include <stdatomic.h>

static _Atomic uint64_t acquire_calls = 0;
static _Atomic uint64_t failed_attempts = 0;
static _Atomic uint64_t max_failed_attempts = 0;

double purchase_transaction(
    int32_t *locks,
    double *warehouse,
    double *lost_sales,
    int64_t address,
    double requested,
    double *available_result,
    double *remaining_result
)
{
    uint64_t failed = 0;
    int32_t expected;

    atomic_fetch_add_explicit(
        &acquire_calls,
        1,
        memory_order_relaxed
    );

    for (;;) {
        expected = 0;

        if (atomic_compare_exchange_weak_explicit(
                (_Atomic int32_t *) &locks[address],
                &expected,
                1,
                memory_order_acquire,
                memory_order_relaxed)) {
            break;
        }

        failed++;

        atomic_fetch_add_explicit(
            &failed_attempts,
            1,
            memory_order_relaxed
        );
    }

    uint64_t current_max = atomic_load_explicit(
        &max_failed_attempts,
        memory_order_relaxed
    );

    while (
        failed > current_max
        && !atomic_compare_exchange_weak_explicit(
            &max_failed_attempts,
            &current_max,
            failed,
            memory_order_relaxed,
            memory_order_relaxed
        )
    ) {
    }

    double available = warehouse[address];

    if (available < 0.0) {
        available = 0.0;
    }

    double purchased = (
        requested < available
        ? requested
        : available
    );

    double remaining = available - purchased;
    double lost = requested - purchased;

    if (purchased > 0.0) {
        warehouse[address] = remaining;
    }

    // All updates for this supplier occur while the same supplier lock is held.
    // lost_sales therefore needs no separate atomic operation.
    if (lost > 0.0) {
        lost_sales[address] += lost;
    }

    atomic_store_explicit(
        (_Atomic int32_t *) &locks[address],
        0,
        memory_order_release
    );

    *available_result = available;
    *remaining_result = remaining;

    return purchased;
}

uint64_t get_acquire_calls(void)
{
    return atomic_load_explicit(
        &acquire_calls,
        memory_order_relaxed
    );
}

uint64_t get_failed_attempts(void)
{
    return atomic_load_explicit(
        &failed_attempts,
        memory_order_relaxed
    );
}

uint64_t get_max_failed_attempts(void)
{
    return atomic_load_explicit(
        &max_failed_attempts,
        memory_order_relaxed
    );
}

void reset_lock_stats(void)
{
    atomic_store_explicit(
        &acquire_calls,
        0,
        memory_order_relaxed
    );

    atomic_store_explicit(
        &failed_attempts,
        0,
        memory_order_relaxed
    );

    atomic_store_explicit(
        &max_failed_attempts,
        0,
        memory_order_relaxed
    );
}
"""

    if sharedComm.Get_rank() == 0:
        sourcePath.write_text(source, encoding="utf-8")

        if platform.system() == "Darwin":
            command = [
                "cc", "-O3", "-std=c11", "-dynamiclib",
                str(sourcePath), "-o", str(libraryPath),
            ]
        else:
            command = [
                "cc", "-O3", "-std=c11", "-shared", "-fPIC",
                str(sourcePath), "-o", str(libraryPath),
            ]

        subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )

    sharedComm.Barrier()

    library = ctypes.CDLL(str(libraryPath))

    library.purchase_transaction.argtypes = [
        ctypes.POINTER(ctypes.c_int32),
        ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_double),
        ctypes.c_int64,
        ctypes.c_double,
        ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_double),
    ]
    library.purchase_transaction.restype = ctypes.c_double

    library.get_acquire_calls.argtypes = []
    library.get_acquire_calls.restype = ctypes.c_uint64

    library.get_failed_attempts.argtypes = []
    library.get_failed_attempts.restype = ctypes.c_uint64

    library.get_max_failed_attempts.argtypes = []
    library.get_max_failed_attempts.restype = ctypes.c_uint64

    library.reset_lock_stats.argtypes = []
    library.reset_lock_stats.restype = None

    return library


class Model:
    
    def __init__(self):
        

        self.count_months = -1
        self.propFactor = 0

        # Firm production types represented remotely by lightweight proxies.
        # Extend this set if additional production types become remote suppliers.
        self.proxyProductionTypes = {"Int", "I"}

        # Detailed timing diagnostics can be disabled from modelEU27.yaml.
        self.detailedTiming = not params.get('stop_detailed_timing', False)
        self.superTechnicalPrint = superTechnicalPrint
        if self.detailedTiming:
            self.phaseTimingWall = {}
            self.phaseTimingCpu = {}
            self.reduceTimingWall = {}
            self.reduceTimingCpu = {}
        
        ##################################### SCHEDULE ########################################
        #######################################################################################
        runner.schedule_event(          0.0,     self.createFirmSubsets)
        if self.detailedTiming: # used to measure how much time each step employs
            runner.schedule_repeating_event(0.1,  1, self.timedSupply)
            runner.schedule_repeating_event(0.2,  1, self.timedDemand)
            # runner.schedule_repeating_event(0.3,  1, self.timedAggregatingResultsByCountry)
            runner.schedule_repeating_event(0.35, 1, self.timedSelectSuppliers)
            runner.schedule_repeating_event(0.355, 1, self.timedAdjustCapital)
            runner.schedule_repeating_event(0.36,  1, self.timedAggregatingResultsByCountry)
            runner.schedule_repeating_event(0.4,  1, self.timedAggregatingRanks)
        else:
            runner.schedule_repeating_event(0.1,  1, self.supply)
            runner.schedule_repeating_event(0.2,  1, self.demand)
            # runner.schedule_repeating_event(0.3,  1, self.aggregatingResultsByCountry)
            runner.schedule_repeating_event(0.35, 1, self.selectSuppliers)
            runner.schedule_repeating_event(0.355, 1, self.adjustCapital)
            runner.schedule_repeating_event(0.36,  1, self.aggregatingResultsByCountry)
            runner.schedule_repeating_event(0.4,  1, self.aggregatingRanks)
        runner.schedule_repeating_event(0.7,  1, self.showResults)

        #runner.schedule_stop(0.8) # N x 12 identifies the number of years to be run as one time-step is one month
        runner.schedule_stop(params['howManyCycles']) # N x 12 identifies the number of years to be run 
                                                      # as one time-step is one month
        runner.schedule_end_event(self.finish)

        self.eu_countries = ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 
                             'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 
                             'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 
                             'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 
                             'Slovenia', 'Spain', 'Sweden', 'totEU27']
        country_numid_list = list(range(27))

        self.selected_countries = params['selected_countries']
    
        eu_productivities = pd.read_pickle("./eu_productivities.xp") # generated in computing_productivity.ipynb
        self.adjustingCoefficient = eu_productivities['no agri tot prod Italy-adjusted'] # adjustment index, based on Istat data on capital 
        self.agri_adjustingCoefficient = eu_productivities['agri tot prod']  
        nama_av = pd.read_pickle("./nama_av.xp") # generated in computing_productivity.ipynb
        self.shareConsumptionDemandTable = pd.read_pickle("./shareConsumptionDemand.xp")  # generated in initialConsumptionDemand.ipynb

        self.lc_countries = pd.read_pickle("./lc_countries.xp") # generated in lc_countries.ipynb
        # lc_countries is a column of the labor cost index by country
        
        self.gdp27_comparison = pd.read_pickle("./gdp27_comparison.xp") # generated in GNP27basicPrices.ipynb
        # at the moment here only for comparison reasons (should not be relevant to new steps of the model)
        
        self.consumptionShares = pd.read_pickle("./shareConsumptionDemand.xp") # generated in initialConsumptionDemand.ipynb

        #print(self.consumptionShares.iloc[2,-2],self.consumptionShares.iloc[2,-1])
        #self.consumptionShares = self.consumptionShares.set_index("country")
        #print(self.consumptionShares.at["Bulgaria", "private_cons / gdp"], self.consumptionShares.at["Bulgaria", "gov_cons /gdp"])

        
        ####################################################################################################
        ###################################### CREATE FIRM AGENTS ##########################################
        ####################################################################################################

        with open('naio_io_N.xp', 'rb') as f:
            intermediateInputs_float64 = pickle.load(f) #numpy array, only numeric, see last row of firm-features-generation.ipynb
                                                #eliminating nan in col 43 (used), not in 64 (never used)
        self.intermediateInputs = intermediateInputs_float64.astype(float).tolist()
                                                # from now onwards we have to call its values by using * [i][j] * as indexes
        
        #firm investment shares
        nama = params['tableOfInvestmentShares']
        print('rank', rank, 'tableOfInvestmentShare', nama)

        if nama==0: self.invShares = pd.read_pickle("./invTableNoNama.xp") #ignoring nama infos
        if nama==1: self.invShares = pd.read_pickle("./invTableNama.xp")   #using nama infos
        if nama==2: self.invShares = pd.read_pickle("./invTableNamaIPF.xp")   #using nama infos

        self.invSharesNp = self.invShares.to_numpy()
        #print(self.invSharesNp)

        #65 rows x 66 columns
        #print (self.invShares)
        #print (self.invShares.iloc[0,0]) #Crop
        #print (self.invShares.iloc[1,0]) #Forest
        #print (self.invShares.iloc[0,1]) #0.002242 values for invTableNoNama (here and following)
        #print (self.invShares.iloc[1,1]) #0.002242
        #print (self.invShares.iloc[0,2]) #0.000131
        #print (self.invShares.iloc[1,2]) #0.000131
        #print (self.invShares.iloc[43,0]) #Imputed 
        #print (self.invShares.iloc[44,0]) #Real
            
        for i in range(64): #64, as rows 0:63
            self.intermediateInputs[i][43]=0
            #self.intermediateInputs[i][64]=0 #col 64 never used, as sector 65 was dropped

        fileName = "ff_with_class_limits.csv" #input("file name? ")

        # how many firms
        #self.simulationFirmsExAnteNumber = int(input("how many firms? "))
        self.simulationFirmsExAnteNumber = params['howManyFirms']
        print('rank', rank, 'how many firms', self.simulationFirmsExAnteNumber)

        with open("workers_distribution.xp", "rb") as f:
            worker_distribution = pickle.load(f)

        scaled_worker_distribution = scalingWorkerDistribution(worker_distribution) 
        L  = [(0,9), (9,19), (19,49), (49,249), (249,1849)] #probabilities
        #scaled_worker_distribution[L[cl][0]:L[cl][1]] to call specific values within a class
        wN = [i for i in range(1,1850)]
        #wN[L[cl][0]:L[cl][1]] to call specific values within a class
        
        # smart capital
        """
        self.smart_capital = 2
        while self.smart_capital != 0 and self.smart_capital != 1:
            self.smart_capital = int(input("use smart capital? (Please, use only 1|0 to express True/False values) "))
            if self.smart_capital != 0 and self.smart_capital != 1: print("Please, use only 1|0 to express True/False values")
        """
        self.smart_capital = params['useSmartCapital']
        print('rank', rank, 'use smart capital', self.smart_capital)

        # from addedvalue_by_country.ipynb - in folder structuralData
        with open('addedvalue_countryshares.xp', 'rb') as f:
            addedvalue_countryshares = pickle.load(f)
            countryshare_list = addedvalue_countryshares['Share in 2020'].tolist()
            # addedvalue_countryshares['Country', 'Share in 2020'] where 'Country' is an index, not a column
            # to call the share use addedvalue_countryshares.iloc[0,x]
            # can also call by country doing addedvalue_countryshares.at['Austria', 'Share in 2020']

        # from sbs_countries.ipynb
        eu_firms_by_employed_number = pd.read_pickle("./eu_firms_by_employed_number.xp") # this is a list of dfs (0-9, 10-19,20-49,50-249,>250)
        sector_special_cases = ['1', '2', '3', '44'] # agri-silvi-fish and imputed rents
        agri_eu27 = pd.read_pickle("./agri_eu27base.xp") # agri-silvi-fish shares (countries are cols to keep 
                                                        # consistency with eu_firms_by_employed_number)
                                                       # the agri_eu27base file comes from ./special_sectors_eu27.ipynb (cell 7)
        agri_eu27.index = ["Agricultural share", "Forestry share", "Fishing share"]

        prob_country_agri = agri_eu27.loc["Agricultural share"].tolist()
        prob_country_silvi = agri_eu27.loc["Forestry share"].tolist()
        prob_country_fishing = agri_eu27.loc["Fishing share"].tolist()

        self.eu27_owners = pd.read_pickle("./eu27_ownership.xp") # 44 sector, imputed rents from houses ownership by EU country
                                                                 # generated from special_sectors_EU27.ipynb

        #importing csv file containing info about firms 
        with open(fileName, newline='') as csvfile:
            firmReader= csv.reader(csvfile, delimiter=',')#, quoting=csv.QUOTE_NONNUMERIC)
    
            self.rowNumber=-1 #to skip the row of the headers
            k=0 
            
            for row in firmReader: #for each record in .csv
                #print(row)
                labor = 0 

                if self.rowNumber >= 0:
                    assignedRanges = (
                        rowAssignments924[
                            self.rowNumber
                        ][rank]
                    )
                else:
                    assignedRanges = []

                if assignedRanges: # skip header and rows with no Firms owned by this rank
                    if row[4]=='': row[4]=0 # it pertains with last rows (naio sector 65)                       
                    sector = row[0]

                    #*****************************************************************************************
                    # Firm counts are computed globally in making924table().
                    # assignedRanges contains stable row-local Firm indices.
                    # The same indices and the same Firm characteristics exist
                    # for every MPI rank count; only ownership changes.
                    #*****************************************************************************************

                    # class attribution
                    if row[3] == "From 0 to 9 persons employed":    cl = 0
                    if row[3] == "From 10 to 19 persons employed":  cl = 1
                    if row[3] == "From 20 to 49 persons employed":  cl = 2
                    if row[3] == "From 50 to 249 persons employed": cl = 3
                    if row[3] == "250 persons employed or more":    cl = 4

                    for rangeStart, rangeStop in assignedRanges:
                        for globalFirmIndex in range(rangeStart, rangeStop):

                            # Each Firm has its own deterministic random stream.
                            # It depends only on the population seed, the 924 row
                            # and the stable Firm index within that row, not on rank.
                            firmRng = np.random.default_rng(
                                np.random.SeedSequence([
                                    firmPopulationSeed,
                                    self.rowNumber,
                                    globalFirmIndex,
                                ])
                            )

                            if row[0] == '1' or row[0] == '2' or row[0] == '3': #agri, silvi and fishing
                                randomizer = firmRng.random()
                                if randomizer <= 0.2: labor = 0
                                elif randomizer <= 0.9 : labor = 1
                                else: labor = 2
                            else:
                                # use L and scaled_worker_distribution
                                labor = firmRng.choice(
                                    wN[L[cl][0]:L[cl][1]],
                                    p=scaled_worker_distribution[L[cl][0]:L[cl][1]],
                                )
                                
                                if sector =='44': labor = 0 # no employees in a virtual sector
                            
                            
                            # assignation of EU country label/number 
                            # non-special cases sectors come from the sbs_countries file,
                            # while agriculture sector is treated in the agri_eu27 file
                            country_counter = 0

                            if not row[0] in sector_special_cases and int(row[0]) != 61 and int(row[0]) != 64:
                                prob_country_sec_cl = eu_firms_by_employed_number[cl].loc[int(row[0])].squeeze().tolist()
                                country_counter = firmRng.choice(
                                    country_numid_list,
                                    p=prob_country_sec_cl,
                                )  
                            
                            elif row[0] == '44': # exactly one imputed-rents Firm per EU27 country
                                country_counter = globalFirmIndex

                            elif int(row[0]) == 61 or int(row[0]) == 64: # 61: membership organizations, 64: services by households
                                country_counter = firmRng.choice(
                                    country_numid_list,
                                    p=countryshare_list,
                                ) 
                                
                            else: # agri forestry and fishing
                                if int(row[0]) == 1:
                                    country_counter = firmRng.choice(country_numid_list, p=prob_country_agri)
                                if int(row[0]) == 2:
                                    country_counter = firmRng.choice(country_numid_list, p=prob_country_silvi)
                                if int(row[0]) == 3:
                                    country_counter = firmRng.choice(country_numid_list, p=prob_country_fishing)

                            capital= float(row[11]) + firmRng.random()*(float(row[12]) - float(row[11]))
                            if self.smart_capital:
                                capital = labor * float(row[10]) # row[10] = recipe 
                                
                            capitalR = float(row[13])
                            wage = float(row[14])

                            # adjusting wage and capital accounting for differences in national productivity
                            # the adjustingCoefficient is a vector containing indeces of national productivity (for agri and no-agri sectors)
                                      # for all the countries (ordered in the standard order in which we report the list of EU countries)
                            
                            if row[0] == '1' or row[0] == '2' or row[0] == '3': #agri, silvi and fishing
                                wage *= self.agri_adjustingCoefficient[country_counter]
                                capital *= self.agri_adjustingCoefficient[country_counter]
                            else:
                                #wage *= self.adjustingCoefficient[country_counter]
                                wage *= self.lc_countries[country_counter]
                                capital *= self.adjustingCoefficient[country_counter]
                            
                            
                            # !!!
                            # for the intermediate inputs each firm must grabble its values directly from the self.intermediateInputs table
                            # by using self.intermediateInputs address which is internal to the model

                            recipe = float(row[10])
                            countryRecipe = recipe * self.adjustingCoefficient[country_counter]
                            
                            if row[0] == '1' or row[0] == '2' or row[0] == '3': 
                                laborProductivity = nama_av['agri_tot_prod'][country_counter]
                            else: 
                                laborProductivity = nama_av['noagri_tot_prod'][country_counter]
                              
                            avgPlannedMarkup = params['plannedMarkup']
                            productionType = row[2]
                            #orderObservationFrequency=rng.integers(row[12], row[13]+1)
                            
                            ################################# adding aFirm ##################################
                            #################################################################################

                            # sector 1-64
                            # country_counter 0-26
                            aFirm =Firm(k, rank, int(sector), labor, capital, capitalR, wage, country_counter, 
                                        recipe, countryRecipe, laborProductivity, avgPlannedMarkup, productionType) 
                                        #orderObservationFrequency
                            # Stable global address: independent of the MPI owner rank.
                            # The same synthetic Firm therefore indexes the same shared
                            # memory slot for every rankNum.
                            aFirm.address = (
                                rowFirmOffsets924[self.rowNumber]
                                + globalFirmIndex
                            )
                            # here we pass "self" as the address of the model to provide all firms with the self.intermediateInputs table
                            
                            context.add(aFirm)
                            k += 1 # first element of the UID of the agents
                            #print(aFirm.uid, aFirm.sector, aFirm.country)

                            # The t=0 intermediate warehouse is initialized later,
                            # after one compact NumPy matrix has been allocated for all
                            # locally owned Firms.
             
                        
                self.rowNumber += 1
                self.simulationFirmsExPostNumber=k #one more, here is a count, not an id
                
            print(self.simulationFirmsExPostNumber, "firms created in rank ", rank)

            


        
        self.countActualExPostFirmNumber = len(list(context.agents(agent_type=0)))

        # Compact persistent intermediate-input inventories. Each rank owns one
        # contiguous float64 matrix instead of three Python lists of 64 objects
        # inside every Firm. Only this warehouse persists across months.
        self.intermediateWarehouseMatrix = np.zeros(
            (self.countActualExPostFirmNumber, 64),
            dtype=np.float64,
        )
        Firm.localIntermediateWarehouse = self.intermediateWarehouseMatrix

        # Compact monthly investment state. These two float64 vectors avoid adding
        # one Python float object per Firm and keep the memory footprint small.
        self.investmentGoodsPurchasedArray = np.zeros(
            self.countActualExPostFirmNumber,
            dtype=np.float64,
        )
        Firm.localInvestmentGoodsPurchased = self.investmentGoodsPurchasedArray

        self.investmentPotentialExportsArray = np.zeros(
            self.countActualExPostFirmNumber,
            dtype=np.float64,
        )
        Firm.localInvestmentPotentialExports = self.investmentPotentialExportsArray

        # Initialize t=0 inventories exactly as before, one Firm at a time, to
        # avoid a large temporary N x 64 allocation.
        for aFirm in context.agents(agent_type=0):
            warehouse = self.intermediateWarehouseMatrix[aFirm.uid[0]]
            estimatedAddedValueTminus1 = aFirm.estimatedAddedValueTminus1
            for j in range(64):
                warehouse[j] = (
                    self.intermediateInputs[j][aFirm.sector - 1]
                    * estimatedAddedValueTminus1
                )
            # This scalar is needed only to initialize t=0 inventories.
            del aFirm.estimatedAddedValueTminus1

        # Shared memory for Firm transactional state.
        self.firmCounts = comm.allgather(self.countActualExPostFirmNumber)
        self.firmOffsets = [0] * rankNum
        for i in range(1, rankNum):
            self.firmOffsets[i] = self.firmOffsets[i-1] + self.firmCounts[i-1]
        self.totalFirmNumber = sum(self.firmCounts)

        self.sharedComm = comm.Split_type(MPI.COMM_TYPE_SHARED, key=rank)
        self.sharedRank = self.sharedComm.Get_rank()
        self.sharedRankNum = self.sharedComm.Get_size()
        if self.sharedRankNum != rankNum:
            raise RuntimeError('All ranks must run on the same shared-memory node')

        floatSize = np.dtype(np.float64).itemsize
        intSize = np.dtype(np.int32).itemsize
        int64Size = np.dtype(np.int64).itemsize
        int16Size = np.dtype(np.int16).itemsize
        int8Size = np.dtype(np.int8).itemsize

        if self.sharedRank == 0:
            outputWarehouseBytes = self.totalFirmNumber * floatSize
            outputWarehouseSnapshotBytes = self.totalFirmNumber * floatSize
            targetAddedValueBytes = self.totalFirmNumber * floatSize
            lostSalesBytes = self.totalFirmNumber * floatSize
            buyerRefillBytes = self.totalFirmNumber * floatSize
            matchedSupplierAddressBytes = self.totalFirmNumber * int64Size
            countryBytes = self.totalFirmNumber * int16Size
            sectorBytes = self.totalFirmNumber * int16Size
            productionTypeCodeBytes = self.totalFirmNumber * int8Size
            supplierLockBytes = self.totalFirmNumber * intSize
        else:
            outputWarehouseBytes = 0
            outputWarehouseSnapshotBytes = 0
            targetAddedValueBytes = 0
            lostSalesBytes = 0
            buyerRefillBytes = 0
            matchedSupplierAddressBytes = 0
            countryBytes = 0
            sectorBytes = 0
            productionTypeCodeBytes = 0
            supplierLockBytes = 0

        self.outputWarehouseWin = MPI.Win.Allocate_shared(outputWarehouseBytes, floatSize, comm=self.sharedComm)
        outputWarehouseBuffer, _ = self.outputWarehouseWin.Shared_query(0)
        Firm.sharedOutputWarehouse = np.ndarray(buffer=outputWarehouseBuffer, dtype=np.float64,
                                                shape=(self.totalFirmNumber,))

        # Shared frozen output used to build each quantity-weighted Two-Tape round.
        # It is refreshed before the first purchase round and before every
        # additional refill round.
        self.outputWarehouseSnapshotWin = MPI.Win.Allocate_shared(
            outputWarehouseSnapshotBytes,
            floatSize,
            comm=self.sharedComm,
        )
        outputWarehouseSnapshotBuffer, _ = self.outputWarehouseSnapshotWin.Shared_query(0)
        Firm.sharedOutputWarehouseSnapshot = np.ndarray(
            buffer=outputWarehouseSnapshotBuffer,
            dtype=np.float64,
            shape=(self.totalFirmNumber,),
        )

        self.targetAddedValueWin = MPI.Win.Allocate_shared(targetAddedValueBytes, floatSize, comm=self.sharedComm)
        targetAddedValueBuffer, _ = self.targetAddedValueWin.Shared_query(0)
        Firm.sharedTargetAddedValue = np.ndarray(buffer=targetAddedValueBuffer, dtype=np.float64,
                                                 shape=(self.totalFirmNumber,))

        # Monthly lost sales by supplier Firm. One float64 per global Firm address.
        # A transaction adds requested - purchased whenever output stock is
        # insufficient. The vector is reset at the start of each monthly search.
        self.lostSalesWin = MPI.Win.Allocate_shared(
            lostSalesBytes,
            floatSize,
            comm=self.sharedComm,
        )
        lostSalesBuffer, _ = self.lostSalesWin.Shared_query(0)
        Firm.sharedLostSales = np.ndarray(
            buffer=lostSalesBuffer,
            dtype=np.float64,
            shape=(self.totalFirmNumber,),
        )

        # One shared refill vector is reused sector by sector. Owners write the
        # current residual refill; country coordinators then build the global
        # buyer tape from the complete vector.
        self.buyerRefillWin = MPI.Win.Allocate_shared(
            buyerRefillBytes,
            floatSize,
            comm=self.sharedComm,
        )
        buyerRefillBuffer, _ = self.buyerRefillWin.Shared_query(0)
        Firm.sharedBuyerRefill = np.ndarray(
            buffer=buyerRefillBuffer,
            dtype=np.float64,
            shape=(self.totalFirmNumber,),
        )

        # Matched supplier global address for the sector currently being
        # processed. Different country coordinators write disjoint buyer slots.
        self.matchedSupplierAddressWin = MPI.Win.Allocate_shared(
            matchedSupplierAddressBytes,
            int64Size,
            comm=self.sharedComm,
        )
        matchedSupplierAddressBuffer, _ = (
            self.matchedSupplierAddressWin.Shared_query(0)
        )
        Firm.sharedMatchedSupplierAddress = np.ndarray(
            buffer=matchedSupplierAddressBuffer,
            dtype=np.int64,
            shape=(self.totalFirmNumber,),
        )

        # Static buyer country metadata, indexed by stable global Firm address.
        self.countryWin = MPI.Win.Allocate_shared(
            countryBytes,
            int16Size,
            comm=self.sharedComm,
        )
        countryBuffer, _ = self.countryWin.Shared_query(0)
        Firm.sharedCountry = np.ndarray(
            buffer=countryBuffer,
            dtype=np.int16,
            shape=(self.totalFirmNumber,),
        )

        # Static sector metadata lets remote FirmProxy objects remain almost
        # empty: only the stable global address is stored in each proxy.
        self.sectorWin = MPI.Win.Allocate_shared(
            sectorBytes,
            int16Size,
            comm=self.sharedComm,
        )
        sectorBuffer, _ = self.sectorWin.Shared_query(0)
        Firm.sharedSector = np.ndarray(
            buffer=sectorBuffer,
            dtype=np.int16,
            shape=(self.totalFirmNumber,),
        )

        # Production type is stored as a compact int8 code in shared memory.
        self.productionTypeCodeWin = MPI.Win.Allocate_shared(
            productionTypeCodeBytes,
            int8Size,
            comm=self.sharedComm,
        )
        productionTypeCodeBuffer, _ = (
            self.productionTypeCodeWin.Shared_query(0)
        )
        Firm.sharedProductionTypeCode = np.ndarray(
            buffer=productionTypeCodeBuffer,
            dtype=np.int8,
            shape=(self.totalFirmNumber,),
        )

        # One int32 lock per Firm, indexed by the global Firm address.
        self.supplierLockWin = MPI.Win.Allocate_shared(
            supplierLockBytes,
            intSize,
            comm=self.sharedComm,
        )

        supplierLockBuffer, _ = self.supplierLockWin.Shared_query(0)

        self.supplierLocks = np.ndarray(
            buffer=supplierLockBuffer,
            dtype=np.int32,
            shape=(self.totalFirmNumber,),
        )

        if self.sharedRank == 0:
            Firm.sharedOutputWarehouse[:] = 0.0
            Firm.sharedOutputWarehouseSnapshot[:] = 0.0
            Firm.sharedTargetAddedValue[:] = 0.0
            Firm.sharedLostSales[:] = 0.0
            Firm.sharedBuyerRefill[:] = 0.0
            Firm.sharedMatchedSupplierAddress[:] = -1
            Firm.sharedCountry[:] = -1
            Firm.sharedSector[:] = -1
            Firm.sharedProductionTypeCode[:] = -1
            self.supplierLocks[:] = 0

        self.sharedComm.Barrier()

        # Compile and load the C11 atomic helper once on this node.
        self.atomicLockLibrary = buildAtomicLockHelper(
            self.sharedComm
        )

        # Cache the shared-array base pointers once. The multirank purchase
        # path can then execute lock, read, update and unlock in one C call.
        self.supplierLocksPointer = ctypes.cast(
            self.supplierLocks.ctypes.data,
            ctypes.POINTER(ctypes.c_int32),
        )

        self.outputWarehousePointer = ctypes.cast(
            Firm.sharedOutputWarehouse.ctypes.data,
            ctypes.POINTER(ctypes.c_double),
        )

        self.lostSalesPointer = ctypes.cast(
            Firm.sharedLostSales.ctypes.data,
            ctypes.POINTER(ctypes.c_double),
        )

        # Reusable C result objects avoid allocating ctypes objects for
        # every supplier transaction.
        self.purchaseAvailableResult = ctypes.c_double()
        self.purchaseRemainingResult = ctypes.c_double()

        self.purchaseAvailablePointer = ctypes.pointer(
            self.purchaseAvailableResult
        )

        self.purchaseRemainingPointer = ctypes.pointer(
            self.purchaseRemainingResult
        )

        # The global address was assigned at Firm creation from the 924 row
        # and row-local Firm index. It is therefore independent of rankNum.
        # Populate static and initial shared state at those stable addresses.
        for aFirm in context.agents(agent_type=0):

            if not 0 <= aFirm.address < self.totalFirmNumber:
                raise RuntimeError(
                    f"Invalid stable Firm address {aFirm.address}"
                )

            Firm.sharedTargetAddedValue[
                aFirm.address
            ] = aFirm._targetAddedValue
            # The fallback scalar is no longer needed once the shared state exists.
            del aFirm._targetAddedValue

            Firm.sharedCountry[
                aFirm.address
            ] = aFirm.country

            Firm.sharedSector[
                aFirm.address
            ] = aFirm.sector

            try:
                productionTypeCode = PRODUCTION_TYPE_TO_CODE[
                    aFirm.productionType
                ]
            except KeyError as exc:
                raise ValueError(
                    f"Unknown Firm productionType {aFirm.productionType!r}"
                ) from exc

            Firm.sharedProductionTypeCode[
                aFirm.address
            ] = productionTypeCode

        self.sharedComm.Barrier()

        if (
            np.any(Firm.sharedCountry < 0)
            or np.any(Firm.sharedSector < 1)
            or np.any(Firm.sharedProductionTypeCode < 0)
        ):
            raise RuntimeError(
                "Stable global Firm metadata contain unassigned slots."
            )

        # No Repast4Py ghosts are created. Remote suppliers will be represented
        # by lightweight FirmProxy objects built from shared static metadata.

        self.AFF_FirmNumber = 10000000
        self.Eu_FirmNumber = 32251874 + self.AFF_FirmNumber
        self.AFF_WorkerNumber = 9000000
        self.Eu_WorkerNumber = 162467679 + 9000000
        self.Eu_EmployeeCompensations = 7447036.79
        self.Eu_GDP_withVAT2022 = 16144780 #https://ec.europa.eu/eurostat/databrowser/view/tec00001/default/table?lang=en
        self.Eu_AddedValue2022 = 14303899 #naio table 2022 (milions) total Added value, gross
        self.Eu_Intermediate2022 = 16939701.18 	 
        if self.propFactor == 0: self.propFactor = self.Eu_FirmNumber / self.countActualExPostFirmNumber
        print("propFactor", self.propFactor,"in rank",rank)

        
    ###### firm subsets ######
    # Timing wrappers for the main monthly phases.
    def timedCall(self, phaseName, method):
        wallStart = time.perf_counter()
        cpuStart = time.process_time()
        method()
        self.phaseTimingWall[phaseName] = time.perf_counter() - wallStart
        self.phaseTimingCpu[phaseName] = time.process_time() - cpuStart


    def timedSupply(self):
        self.timedCall("supply", self.supply)


    def timedDemand(self):
        self.timedCall("demand", self.demand)


    def timedAdjustCapital(self):
        self.timedCall("capital", self.adjustCapital)


    def timedAggregatingResultsByCountry(self):
        self.timedCall("country", self.aggregatingResultsByCountry)


    def timedSelectSuppliers(self):

        self.resetSupplierLockStats()

        self.timedCall(
            "suppliers",
            self.selectSuppliers
        )

        self.printSupplierLockStats()


    def timedAggregatingRanks(self):
        self.reduceTimingWall = {}
        self.reduceTimingCpu = {}
        self.timedCall("ranks", self.aggregatingRanks)

        if self.superTechnicalPrint:
            print(
                f"TIMING rank {rank:3d} month {self.count_months + 1:3d} | "
                f"wall: supply {self.phaseTimingWall.get('supply', 0.0):.6f}  "
                f"demand {self.phaseTimingWall.get('demand', 0.0):.6f}  "
                f"capital {self.phaseTimingWall.get('capital', 0.0):.6f}  "
                f"country {self.phaseTimingWall.get('country', 0.0):.6f}  "
                f"suppliers {self.phaseTimingWall.get('suppliers', 0.0):.6f}  "
                f"ranks {self.phaseTimingWall.get('ranks', 0.0):.6f}  "
                f"pack {self.reduceTimingWall.get('pack', 0.0):.6f}  "
                f"reduce {self.reduceTimingWall.get('reduce', 0.0):.6f}  "
                f"unpack {self.reduceTimingWall.get('unpack', 0.0):.6f} | "
                f"cpu: supply {self.phaseTimingCpu.get('supply', 0.0):.6f}  "
                f"demand {self.phaseTimingCpu.get('demand', 0.0):.6f}  "
                f"capital {self.phaseTimingCpu.get('capital', 0.0):.6f}  "
                f"country {self.phaseTimingCpu.get('country', 0.0):.6f}  "
                f"suppliers {self.phaseTimingCpu.get('suppliers', 0.0):.6f}  "
                f"ranks {self.phaseTimingCpu.get('ranks', 0.0):.6f}  "
                f"pack {self.reduceTimingCpu.get('pack', 0.0):.6f}  "
                f"reduce {self.reduceTimingCpu.get('reduce', 0.0):.6f}  "
                f"unpack {self.reduceTimingCpu.get('unpack', 0.0):.6f}",
                flush=True,
            )


    def createFirmSubsets(self):
        
        # creating all firms database by sector, one global and one for each productionType
        self.firmListBySector =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeC   =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeI   =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeInt =  [[] for _ in range(64)]  

        for aFirm in context.agents(agent_type=0):
            self.firmListBySector[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="C":   self.firmListBySectorWithProductionTypeC[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="I":   self.firmListBySectorWithProductionTypeI[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="Int": self.firmListBySectorWithProductionTypeInt[aFirm.sector-1].append(aFirm)

        # creating all firms database by country
        self.firmListByCountry =  [[] for _ in range(27)]

        for aFirm in context.agents(agent_type=0):
            self.firmListByCountry[aFirm.country].append(aFirm) 

    
        # Build lightweight proxies only for remote production types actually
        # needed on this rank. FirmProxy stores only a stable global address;
        # country, sector, production type and dynamic supplier state are read
        # directly from shared arrays.
        localFirmByAddress = {
            aFirm.address: aFirm
            for aFirm in context.agents(agent_type=0)
        }

        firmProxyCount = 0
        firstFirmProxy = None

        for productionType in self.proxyProductionTypes:

            try:
                productionTypeCode = PRODUCTION_TYPE_TO_CODE[
                    productionType
                ]
            except KeyError as exc:
                raise ValueError(
                    f"Unknown proxy production type {productionType!r}"
                ) from exc

            proxyCandidateAddresses = np.flatnonzero(
                Firm.sharedProductionTypeCode
                == productionTypeCode
            )

            for addressValue in proxyCandidateAddresses:
                address = int(addressValue)

                # Locally owned Firms remain the full active Firm objects.
                if address in localFirmByAddress:
                    continue

                proxy = FirmProxy(address)

                if firstFirmProxy is None:
                    firstFirmProxy = proxy

                firmProxyCount += 1
                sectorIndex = proxy.sector - 1

                # Proxies are needed only on supplier-side subsets. The generic
                # sector list is intentionally left local because it is not used
                # by cross-rank supplier matching.
                if productionType == "C":
                    self.firmListBySectorWithProductionTypeC[
                        sectorIndex
                    ].append(proxy)
                elif productionType == "I":
                    self.firmListBySectorWithProductionTypeI[
                        sectorIndex
                    ].append(proxy)
                elif productionType == "Int":
                    self.firmListBySectorWithProductionTypeInt[
                        sectorIndex
                    ].append(proxy)

        # Report the shallow proxy footprint. Shared NumPy metadata are allocated
        # only once per node and are not replicated inside each proxy object.
        proxyObjectBytes = (
            sys.getsizeof(firstFirmProxy)
            if firstFirmProxy is not None
            else 0
        )
        proxyStats = comm.gather(
            (firmProxyCount, proxyObjectBytes),
            root=0,
        )

        if rank == 0:
            proxyCounts = [
                item[0]
                for item in proxyStats
            ]
            proxySizes = [
                item[1]
                for item in proxyStats
                if item[1] > 0
            ]
            proxySize = (
                proxySizes[0]
                if proxySizes
                else 0
            )
            print(
                "FirmProxy remote suppliers per rank "
                f"min {min(proxyCounts)}, max {max(proxyCounts)}, "
                f"shallow object size {proxySize} bytes",
                flush=True,
            )

        # Global Two-Tape requires exactly the same supplier ordering on every
        # rank. Stable global addresses make this ordering rank-invariant.
        for supplierLists in (
            self.firmListBySectorWithProductionTypeInt,
            self.firmListBySectorWithProductionTypeI,
        ):
            for sectorSuppliers in supplierLists:
                sectorSuppliers.sort(
                    key=lambda supplier: supplier.address
                )

        self.localFirms = list(
            localFirmByAddress.values()
        )

        self.supplierByAddress = {
            supplier.address: supplier
            for supplierLists in (
                self.firmListBySectorWithProductionTypeInt,
                self.firmListBySectorWithProductionTypeI,
            )
            for sectorSuppliers in supplierLists
            for supplier in sectorSuppliers
        }

        # Every rank sees the same stable global buyer order for each country.
        self.globalBuyerAddressesByCountry = [
            np.flatnonzero(
                Firm.sharedCountry == country
            ).astype(np.int64)
            for country in range(27)
        ]

        
        if rank == 0: print(f"Rank {rank} Starting phase concluded: global time {T()}, cpu time {Tc()}")


    # Remote suppliers have already been represented by FirmProxy objects.
    # supply 
    def supply(self):

        self.count_months += 1
        
        # here we use 28 but the last positiion of the vector can be an empty space 
        
        self.countWorkers = [0] * 28
        self.countEmployeeCompensations = [0] * 28
        self.countCapitalR = [0] * 28
        self.countActualAddedValue = [0] * 28
        self.countAFF_Firms = [0] * 28
        self.countAFF_WorkerNumber = [0] * 28
        self.intermediateTotalSupplyBySector = [0] * 64
        self.intermediateTotalSupplyByCountry = [0] * 28
        self.intermediateTotalSupplyBySectorAndCountry =  [[0] * 28 for _ in range(64)]
        self.investmentTotalSupplyBySector =  [0] * 64
        self.investmentTotalSupplyByCountry =  [0] * 28
        self.investmentTotalSupplyBySectorAndCountry = [[0] * 28 for _ in range(64)]
        self.intBySectors=[0]*64
        self.addValBySectors=[0]*64
        self.markupTentative= params['plannedMarkup']


        for aFirm in context.agents(agent_type=0):
      
        ###################################################################################################
        ########################### COMPUTING ADDED VALUE BY FIRM IN EACH RANK ############################
        ###################################################################################################
            
            self.countWorkers[aFirm.country] += aFirm.labor
            self.countEmployeeCompensations[aFirm.country] += (aFirm.labor*aFirm.wage) / 12
            self.countCapitalR[aFirm.country] += (aFirm.capital*aFirm.capitalR) / 12
            if int(aFirm.sector) <= 3: 
                self.countAFF_Firms[aFirm.country] += 1
                self.countAFF_WorkerNumber[aFirm.country] +=aFirm.labor
                
            aFirm.targetAddedValue= float(aFirm.labor * aFirm.wage + aFirm.capital * 1000 * aFirm.capitalR) / 12   # float to avoid np.float64 in vector
            aFirm.targetAddedValue *= self.markupTentative 

            
    
            # firms account for the availability of EACH intermediate good in the intermediate warehouse to produce
            # we have to consider the quantity (in value) that we need for each intermediate good to produce our desired added value
            
            # if this quantity is not available (or partially not available) the firm will limit the production to the quantity 
                                                                                              # of intermediate goods available
            # for simplicity, we consider for this purpose only intermediate goods evaluated at least at 1% of the firm added value
                
            # notice that firms usually use multiple intermediate goods
            # if many of them are unavailable we have to consider the lower quantity available for the firm production in that cycle 
            # worst case scenario q of one intermediate good = 0 ---> production = 0
        
            
            aFirm.actualAddedValue = aFirm.targetAddedValue
            warehouse = aFirm.intermediateWarehouse
            for j in range(64):
            # what is doing: cutting actualAddedValue
                if warehouse[j] < (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])* 0.99\
                                                and self.intermediateInputs[j][aFirm.sector-1] > 0.01:
                    aFirm.actualAddedValue *= warehouse[j]/ (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
            
            # after determining aFirm.actualAddedValue across all 64 sectors we can withdraw the corresponding 
                                                        # intermediate goods quotas from their warehouse

            #if aFirm.targetAddedValue - aFirm.actualAddedValue > 0.001: print(aFirm.uid, aFirm.sector-1)


            # sum of all the actual added values
            self.countActualAddedValue[aFirm.country] += aFirm.actualAddedValue

            aFirm.intermediateWarehouseWithdraw = 0
            for j in range(64):
                warehouse[j] -= (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
                aFirm.intermediateWarehouseWithdraw += (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
            
            ############################ intermediate goods firm production ###############################
            
            if aFirm.productionType == "Int": 
                # totalizers to measure this quantity in a matrix sector x country
                self.intermediateTotalSupplyBySector[aFirm.sector-1] +=  aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw
                self.intermediateTotalSupplyByCountry[aFirm.country] +=  aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw
                self.intermediateTotalSupplyBySectorAndCountry[j][aFirm.country] += aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw


            

            ############################ investment goods firm production ##################################
            
            if aFirm.productionType == "I":
                self.investmentTotalSupplyBySector[aFirm.sector-1] +=  aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw
                self.investmentTotalSupplyByCountry[aFirm.country] +=  aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw
                self.investmentTotalSupplyBySectorAndCountry[j][aFirm.country] += aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw


            
        ###################################################################################################
        ###################### SENDING PRODUCTS TO WAREHOUSES BY FIRM IN EACH RANK ########################
        ###################################################################################################
            aFirm.outputWarehouse += aFirm.actualAddedValue + aFirm.intermediateWarehouseWithdraw 
            # must include a) intermediate goods employed in production; b) imports by producing firm/sector
            
        #print("Supply", self.intermediateTotalSupplyBySector)                
        


    def demand(self): 

        self.firmSubstitutions= [0] * 28
        self.substitutionRate =0.1
        self.substitutionRateL=0.01667 #for sectors 44 and 45, 60 years of duration
        #self.allfirmSubstitutionsByVendorSector=[0]*65
        self.allfirmSubstitutionsByVendorSector=np.zeros([65,28])
        self.allfirmSubstitutionsByBuyerSector=np.zeros([65,28])
        self.intermediateTotalDemandByCountry = [0] * 28
        self.intermediateTotalDemandBySector = [0] * 64
        self.intermediateTotalDemandBySectorAndCountry =  [[0] * 28 for _ in range(64)]
        
        for aFirm in context.agents(agent_type=0):
    
        ###################################################################################################
        ############### CAPITAL SUBSTITUIONS GOODS PURCHASING ORDERS BY FIRM IN EACH RANK #################
        ###################################################################################################
        # here what matters is the substitutions classified by buyer
        # those classified by vendor are relevant only for statistical purposes for check
        
            # imputed rents (special case)
            if aFirm.sector == 44: 
                # from ff_with_class_limits, we use the number of the sector not the position!
                aFirm.capital= (30_000_000 * 1000 * self.eu27_owners.iloc[-1,aFirm.country]) 
                # we change the value of capital only for sector 44, for the other sectors it keeps its value from ff_with_class-limits
                
                # the whole EU capital of the sector, to be subdivided by countries
                # see structuralData > firm-features-generation for explanation, cell "Imputed rents special case"
                self.firmSubstitutions[aFirm.country] += ( ((aFirm.capital*self.substitutionRateL)/self.Eu_FirmNumber)\
                                                           *self.simulationFirmsExAnteNumber) / 12
                                        # it will be reported to the EU scale
                    
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += ( ((aFirm.capital*self.substitutionRateL)\
                                                                         /self.Eu_FirmNumber)*self.simulationFirmsExAnteNumber) / 12

                # buying uniquely from constructions (ie sector 27!)
                self.allfirmSubstitutionsByVendorSector[27-1, aFirm.country]+= ( (aFirm.capital*self.substitutionRateL/self.Eu_FirmNumber)*\
                                                                                                  self.simulationFirmsExAnteNumber) / 12

            # real estate (special case 2)
            elif aFirm.sector == 45: # from ff_with_class_limits, we use the number of sector not the position!
                self.firmSubstitutions[aFirm.country] +=  (aFirm.capital*self.substitutionRateL) / 12
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += (aFirm.capital*self.substitutionRateL) / 12
                    
                for s in range(1,66):
                    self.allfirmSubstitutionsByVendorSector[s-1, aFirm.country]+= (aFirm.capital*self.substitutionRateL*\
                                                                                   self.invSharesNp[aFirm.sector-1,s]) / 12
            
            # general case
            else:                                        
                self.firmSubstitutions[aFirm.country] += (aFirm.capital*self.substitutionRate) / 12
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += (aFirm.capital*self.substitutionRate) / 12

                for s in range(1,66):
                    self.allfirmSubstitutionsByVendorSector[s-1, aFirm.country]+= (aFirm.capital*self.substitutionRate*\
                                                                                 self.invSharesNp[aFirm.sector-1,s]) / 12


        ###################################################################################################
        ###################### INTERMEDIATE GOODS PURCHASING NEEDS BY FIRM IN EACH RANK ###################
        ###################################################################################################

        # this step identifies the desired quantities of intermediate inputs in the case that firms can actually produce their entire added value
        # these quantities will be further compared with the availability of the intermediate inputs
        

            warehouse = aFirm.intermediateWarehouse
            for s in range(64):
                refill = (
                    aFirm.targetAddedValue
                    * self.intermediateInputs[s][aFirm.sector - 1]
                    - warehouse[s]
                )
                # totalizers to measure this quantity in a matrix sector x country
                self.intermediateTotalDemandBySector[s] += refill
                self.intermediateTotalDemandByCountry[aFirm.country] += refill
                self.intermediateTotalDemandBySectorAndCountry[s][aFirm.country] += refill

                
        #print("Demand", self.intermediateTotalDemandBySector)

        # we compared the results of the model computing both the intermediate goods demand and the supply (computed either by sector and by country)
        # follow Intermediate Goods Consistency test at https://prsm.uk/prsm.html?room=UTZ-JPY-SWQ-LJH
        # remember that quantities have been adjusted for the propFactor and from months to year ( x12)
        # the difference between supply and demand of Intermediate goods represents its Export
        

        
        ###################################################################################################
        ############################## COMPUTING GDP BY COUNTRY BY RANK ################################### 
        ###################################################################################################
        # from now we are out of the aFirm "for" cycle because we are observing aggregate consumptions

        self.gdp_eu27 =[0] * 27
        for aCountry in range(27):
            self.gdp_eu27[aCountry] = int(self.markupTentative *\
                                         (self.countEmployeeCompensations[aCountry] * self.propFactor / 1000000\
                                          + self.countCapitalR[aCountry] * self.propFactor / 1000))
            #self.markupTentative * (self.countEmployeeCompensations[aCountry] + self.countCapitalR[aCountry])

        # for jjj in range (27):
            # print(jjj+1,self.eu_countries[jjj],self.gdp_eu27[jjj])


        ###################################################################################################
        ####################### EVALUATING PRIVATE AND GOVERNMENT CONSUMPTIONS ############################ 
        ###################################################################################################
        
        self.private_consumption =[0] * 27
        self.gov_consumption =[0] * 27


        adjusting_consumptions_with_d_imports_private = 1 - 476/7284
        adjusting_consumptions_with_d_imports_gov = 1 - 44/3375
        # these values are copied and rounded from table naio 2022 (data_import.ipynb) to get the parameters providing the adjustment 

        for aCountry in range(27):
            self.private_consumption[aCountry] = self.consumptionShares.iloc[aCountry, -2] * self.gdp_eu27[aCountry]  
            #* adjusting_consumptions_with_d_imports_private ## dubbio: eliminiamo le importazioni dirette per ordinare la prod alle firm?
            self.gov_consumption[aCountry] = self.consumptionShares.iloc[aCountry , - 1] * self.gdp_eu27[aCountry]  
            #* adjusting_consumptions_with_d_imports_gov

        # illuminazione(??): quello che ordiniamo alle imprese produttrici di beni di consumo è la fetta di added vaoue prodotto da loro

        # to calculate internal consumption we introduce two adjusting_consumptions_with_d_imports parameters 
        # neglecting the consumption goods which are directly exported

        # notice that the results produced by the model are very consistent among them
        # let us consider Austria as an example: 
        # simulated gdp = 424188; eurostat gdp (from self.consumptionShares) = 449382.2 --> ratio = 0.94
        # sim private consumption = 16833 * 12 =  201996 ; eurostat private consumption (from self.consumptionShares) = 228963.2 -->  ratio = 0.94
        # meaning that they are perfectly in fit reminding that direct exports are a share included in the general private consumption

        # lastly, notice that direct exports may represent significant tools for policy decisions
                

    ###################################################################################################
    ########################## INTERMEDIATE GOODS RANDOM SUPPLIER SELECTION ###########################
    ###################################################################################################
    def resetSupplierLockStats(self):

        if rankNum == 1:
            return

        self.atomicLockLibrary.reset_lock_stats()


    def printSupplierLockStats(self):

        if rankNum == 1 or not self.superTechnicalPrint:
            return

        acquireCalls = int(
            self.atomicLockLibrary.get_acquire_calls()
        )

        failedAttempts = int(
            self.atomicLockLibrary.get_failed_attempts()
        )

        maxFailedAttempts = int(
            self.atomicLockLibrary.get_max_failed_attempts()
        )

        failedPerAcquire = (
            failedAttempts / acquireCalls
            if acquireCalls > 0
            else 0.0
        )

        print(
            f"LOCK STATS rank {rank} "
            f"month {(self.count_months % 12) + 1} | "
            f"acquires {acquireCalls} "
            f"failed_CAS {failedAttempts} "
            f"failed_per_acquire {failedPerAcquire:.6f} "
            f"max_failed_before_acquire {maxFailedAttempts}"
        )

    def purchaseFromSupplier(self, supplier, requested):

        # Fast path for a single rank: no concurrent access is possible.
        if rankNum == 1:

            available = max(
                0.0,
                supplier.outputWarehouse,
            )

            purchased = min(
                requested,
                available,
            )

            remaining = available - purchased
            lost = requested - purchased

            if purchased > 0.0:
                supplier.outputWarehouse = remaining

            if lost > 0.0:
                Firm.sharedLostSales[supplier.address] += lost

            return purchased, available, remaining

        # Multirank path: lock, warehouse read, purchase, warehouse update
        # and unlock are executed atomically inside one C call.
        purchased = self.atomicLockLibrary.purchase_transaction(
            self.supplierLocksPointer,
            self.outputWarehousePointer,
            self.lostSalesPointer,
            supplier.address,
            requested,
            self.purchaseAvailablePointer,
            self.purchaseRemainingPointer,
        )

        available = self.purchaseAvailableResult.value
        remaining = self.purchaseRemainingResult.value

        return purchased, available, remaining


    def investmentDemandFromSupplierSector(self, aFirm, supplierSector):
        """Return monthly replacement-investment demand in outputWarehouse units."""

        # Firm.capital is stored in thousands, while outputWarehouse is in euros.
        # The factor 1000 therefore puts investment orders in the same units as
        # investment-goods supply before the Two-Tape matching.
        if aFirm.sector == 44:
            # Imputed rents: one virtual Firm per country represents the full
            # national sector. Demand is scaled down to the simulated economy and
            # purchases are scaled back up in adjustCapital(). It buys only from
            # construction, NAIO sector 27 (zero-based supplierSector 26).
            if supplierSector != 26:
                return 0.0

            replacementCapital = (
                (
                    aFirm.capital
                    * self.substitutionRateL
                    / self.Eu_FirmNumber
                )
                * self.simulationFirmsExAnteNumber
                / 12
            )
            share = 1.0

        elif aFirm.sector == 45:
            # Preserve the existing demand() rule for real-estate activities.
            replacementCapital = (
                aFirm.capital
                * self.substitutionRateL
                / 12
            )
            share = float(
                self.invSharesNp[
                    aFirm.sector - 1,
                    supplierSector + 1,
                ]
            )

        else:
            replacementCapital = (
                aFirm.capital
                * self.substitutionRate
                / 12
            )
            share = float(
                self.invSharesNp[
                    aFirm.sector - 1,
                    supplierSector + 1,
                ]
            )

        if not np.isfinite(share) or share < 0.0:
            raise ValueError(
                "Investment shares must be finite and non-negative."
            )

        return max(
            0.0,
            replacementCapital * share * 1000.0,
        )


    def selectInvestmentSuppliers(
        self,
        local_preference,
        total_attempts,
        full_attempts,
        fractionalEligibleLocal,
        fractionalCoordinatorBuyers,
        country_order,
        global_matching_seed,
        period_index,
        import_enabled,
    ):
        """Buy replacement investment goods with the global Two-Tape algorithm."""

        investmentImports = [0.0] * 64

        countTotalInvestmentOperations = 0
        countSuccessfulInvestmentOperations = 0
        countTotalLocalInvestmentOperations = 0
        countSuccessfulLocalInvestmentOperations = 0

        # Purchases are accumulated in capital units (thousands), while the shared
        # refill vector and supplier warehouses remain in output units (euros).
        self.investmentGoodsPurchasedArray.fill(0.0)

        for aSector, sector_suppliers in enumerate(
            self.firmListBySectorWithProductionTypeI
        ):

            # Every owner publishes the current replacement demand of its Firms for
            # this investment-goods supplier sector. The shared vector is reused
            # sector by sector, exactly as for intermediate inputs.
            for aFirm in self.localFirms:
                Firm.sharedBuyerRefill[aFirm.address] = (
                    self.investmentDemandFromSupplierSector(
                        aFirm,
                        aSector,
                    )
                )

            self.sharedComm.Barrier()

            for attempt_index in range(total_attempts):

                if attempt_index > 0:
                    self.sharedComm.Barrier()

                    if self.sharedRank == 0:
                        Firm.sharedOutputWarehouseSnapshot[:] = (
                            Firm.sharedOutputWarehouse
                        )

                    self.sharedComm.Barrier()

                fractional_attempt = (
                    attempt_index >= full_attempts
                )

                for country_position, country_value in enumerate(
                    country_order
                ):

                    if country_position % rankNum != rank:
                        continue

                    aCountry = int(country_value)
                    all_buyer_addresses = (
                        self.globalBuyerAddressesByCountry[aCountry]
                    )

                    if all_buyer_addresses.size == 0:
                        continue

                    Firm.sharedMatchedSupplierAddress[
                        all_buyer_addresses
                    ] = -1

                    if fractional_attempt:
                        buyer_addresses = fractionalCoordinatorBuyers[aCountry]
                    else:
                        buyer_addresses = all_buyer_addresses

                    if buyer_addresses.size == 0:
                        continue

                    if not sector_suppliers:
                        continue

                    # The final seed component separates investment matching from
                    # intermediate-input matching while remaining rank-invariant.
                    match_rng = np.random.default_rng(
                        np.random.SeedSequence([
                            global_matching_seed,
                            period_index + 1,
                            attempt_index,
                            aSector,
                            aCountry,
                            9101,
                        ])
                    )

                    Firm.sharedMatchedSupplierAddress[
                        buyer_addresses
                    ] = match_global_buyer_addresses_to_suppliers(
                        buyer_addresses=buyer_addresses,
                        suppliers=sector_suppliers,
                        buyer_country=aCountry,
                        mult=local_preference,
                        rng=match_rng,
                        supplier_output_snapshot=(
                            Firm.sharedOutputWarehouseSnapshot
                        ),
                        buyer_refill_snapshot=(
                            Firm.sharedBuyerRefill
                        ),
                        point_mode="systematic",
                    )

                self.sharedComm.Barrier()

                for country_value in country_order:
                    aCountry = int(country_value)

                    for aBuyer in self.firmListByCountry[aCountry]:

                        if (
                            fractional_attempt
                            and not fractionalEligibleLocal[aBuyer.uid[0]]
                        ):
                            continue

                        refill = float(
                            Firm.sharedBuyerRefill[aBuyer.address]
                        )

                        if refill <= 0.0:
                            continue

                        supplier_address = int(
                            Firm.sharedMatchedSupplierAddress[
                                aBuyer.address
                            ]
                        )

                        if supplier_address < 0:
                            continue

                        supplier = self.supplierByAddress[
                            supplier_address
                        ]

                        is_local = (
                            supplier.country == aCountry
                        )

                        countTotalInvestmentOperations += 1
                        if is_local:
                            countTotalLocalInvestmentOperations += 1

                        purchased, available, remaining = self.purchaseFromSupplier(
                            supplier,
                            refill,
                        )

                        if purchased > 0.0:
                            countSuccessfulInvestmentOperations += 1
                            if is_local:
                                countSuccessfulLocalInvestmentOperations += 1

                            self.investmentGoodsPurchasedArray[
                                aBuyer.uid[0]
                            ] += purchased / 1000.0
                            refill -= purchased

                        Firm.sharedBuyerRefill[
                            aBuyer.address
                        ] = max(0.0, refill)

            # All ranks finish this supplier sector before residual investment
            # demand is converted into extra-EU imports.
            self.sharedComm.Barrier()

            for aFirm in self.localFirms:
                refill = float(
                    Firm.sharedBuyerRefill[aFirm.address]
                )

                if refill > 0.0 and import_enabled:
                    self.investmentGoodsPurchasedArray[
                        aFirm.uid[0]
                    ] += refill / 1000.0

                    investmentImports[aSector] += refill
                    self.investmentTotalImportsByCountryAndSector[
                        aSector + 1,
                        aFirm.country,
                    ] += refill

                    Firm.sharedBuyerRefill[aFirm.address] = 0.0

        if not params["stop_technical_print"]:
            print(
                "\nInvestment goods imports",
                [
                    (
                        sector + 1,
                        round(value / 1_000_000, 2),
                    )
                    for sector, value in enumerate(investmentImports)
                ],
            )
            print(
                "\nTotal investment goods imports",
                round(sum(investmentImports) / 1_000_000, 2),
            )

            failedInvestmentOperations = (
                countTotalInvestmentOperations
                - countSuccessfulInvestmentOperations
            )
            failedLocalInvestmentOperations = (
                countTotalLocalInvestmentOperations
                - countSuccessfulLocalInvestmentOperations
            )
            print(
                "\ncountTotalInvestmentOperations",
                countTotalInvestmentOperations,
                "countTotalLocalInvestmentOperations",
                countTotalLocalInvestmentOperations,
                "\ncountSuccessfulInvestmentOperations",
                countSuccessfulInvestmentOperations,
                "countSuccessfulLocalInvestmentOperations",
                countSuccessfulLocalInvestmentOperations,
                "\ncountFailedInvestmentOperations",
                failedInvestmentOperations,
                "countFailedLocalInvestmentOperations",
                failedLocalInvestmentOperations,
                flush=True,
            )

        return investmentImports


    def selectSuppliers(self):

        # Diagnostic timing: separate useful supplier work from collective waits.
        supplierDiagTotalWallStart = time.perf_counter()
        supplierDiagTotalCpuStart = time.process_time()

        supplierDiagWallStart = time.perf_counter()
        supplierDiagCpuStart = time.process_time()

        # All ranks complete production and demand before cross-rank purchases start.
        self.sharedComm.Barrier()

        # Freeze the current-month supplier output before any rank can purchase
        # and reset the monthly lost-sales measure.
        if self.sharedRank == 0:
            Firm.sharedOutputWarehouseSnapshot[:] = (
                Firm.sharedOutputWarehouse
            )
            Firm.sharedLostSales[:] = 0.0

        self.sharedComm.Barrier()

        supplierDiagStartBarrierWall = (
            time.perf_counter()
            - supplierDiagWallStart
        )

        supplierDiagStartBarrierCpu = (
            time.process_time()
            - supplierDiagCpuStart
        )

        supplierDiagWorkWallStart = time.perf_counter()
        supplierDiagWorkCpuStart = time.process_time()

        # imports by country and sector
        self.intermediateTotalImportsByCountryAndSector = np.zeros([65,28])

        # potential exports by country and sector
        self.intermediateTotalPotentialExportsByCountryAndSector = np.zeros([65,28])

        # Investment-goods imports and potential exports use NAIO sector numbers
        # 1..64; position 0 is intentionally left empty.
        self.investmentTotalImportsByCountryAndSector = np.zeros([65,28])
        self.investmentTotalPotentialExportsByCountryAndSector = np.zeros([65,28])
        self.investmentGoodsPurchasedArray.fill(0.0)
        self.investmentPotentialExportsArray.fill(0.0)

        # Statistics
        countSuccessfulRefillOperations = 0
        countSuccessfulLocalRefillOperations = 0
        countTotalRefillOperations = 0
        countTotalLocalRefillOperations = 0

        if not params["stop_technical_print"]:
            print("\nBefore refill")
            printDemandOfferRefill(outAddedV=True, intermediateInputs=self.intermediateInputs)

        imports = [0.0] * 64

        if params["restocking"] == "n":
            print("\nRestocking disabled")
            return

        local_preference = float(
            params["localPreference"]
        )

        refill_repeat = float(
            params["max_refill_repeat"]
        )

        if (
            not np.isfinite(refill_repeat)
            or refill_repeat < 0.0
        ):
            raise ValueError(
                "max_refill_repeat must be a finite non-negative number."
            )

        full_refill_repeats = int(
            np.floor(refill_repeat)
        )

        fractional_refill_probability = (
            refill_repeat
            - full_refill_repeats
        )

        if fractional_refill_probability < 1.0e-12:
            fractional_refill_probability = 0.0

        full_attempts = (
            full_refill_repeats + 1
        )

        total_attempts = (
            full_attempts
            + int(fractional_refill_probability > 0.0)
        )

        import_enabled = bool(
            params["import"]
        )

        randomize_country_order = bool(
            params.get(
                "randomize_country_order",
                False,
            )
        )

        if local_preference <= 0.0:
            raise ValueError(
                "localPreference must be strictly positive."
            )

        # One common seed controls the matching realization on every rank.
        # It is deliberately independent of params["myRandom.seed"][rank].
        global_matching_seed = int(
            params.get(
                "globalMatchingSeed",
                params["myRandom.seed"][0],
            )
        )

        period_index = int(self.count_months)

        if randomize_country_order:

            country_rng = np.random.default_rng(
                np.random.SeedSequence([
                    global_matching_seed,
                    period_index + 1,
                    7001,
                ])
            )

            country_order = country_rng.permutation(27)

        else:

            country_order = np.arange(27)

        if self.superTechnicalPrint:
            print(
                "\nGlobal Two-Tape country processing order",
                [
                    int(country)
                    for country in country_order
                ],
            )

        # The fractional round remains Firm-specific and rank-invariant. Store
        # local eligibility in one compact boolean vector; coordinators cache only
        # the selected global buyer addresses for the countries they own.
        fractionalEligibleLocal = None
        fractionalCoordinatorBuyers = {}

        if fractional_refill_probability > 0.0:
            fractionalEligibleLocal = np.zeros(
                self.countActualExPostFirmNumber,
                dtype=np.bool_,
            )

            for aFirm in self.localFirms:
                fractionalEligibleLocal[aFirm.uid[0]] = (
                    deterministic_firm_uniform(
                        aFirm.address,
                        period_index,
                        global_matching_seed,
                    ) < fractional_refill_probability
                )

            for country_position, country_value in enumerate(
                country_order
            ):
                if country_position % rankNum != rank:
                    continue

                aCountry = int(country_value)
                buyer_addresses = self.globalBuyerAddressesByCountry[aCountry]

                if buyer_addresses.size == 0:
                    fractionalCoordinatorBuyers[aCountry] = buyer_addresses
                    continue

                selected = np.fromiter(
                    (
                        deterministic_firm_uniform(
                            int(address),
                            period_index,
                            global_matching_seed,
                        ) < fractional_refill_probability
                        for address in buyer_addresses
                    ),
                    dtype=np.bool_,
                    count=buyer_addresses.size,
                )
                fractionalCoordinatorBuyers[aCountry] = (
                    buyer_addresses[selected]
                )

        # Process one intermediate-input sector at a time. The 64 sectors are
        # economically independent during supplier matching because each supplier
        # belongs to exactly one sector. This loop order lets one shared N-vector
        # hold the complete residual refill state for the current sector only.
        for aSector, sector_suppliers in enumerate(
            self.firmListBySectorWithProductionTypeInt
        ):

            # Reconstruct this sector's initial refill exactly from target demand
            # and the persistent intermediate warehouse. Every owner writes all
            # of its slots, including zero, so no stale state survives.
            for aFirm in self.localFirms:
                refill = (
                    aFirm.targetAddedValue
                    * self.intermediateInputs[aSector][aFirm.sector - 1]
                    - aFirm.intermediateWarehouse[aSector]
                )
                Firm.sharedBuyerRefill[aFirm.address] = max(0.0, refill)

            self.sharedComm.Barrier()

            for attempt_index in range(total_attempts):

                if attempt_index > 0:
                    # The previous round for this sector must finish before the
                    # common supplier-output snapshot is refreshed. Other sectors
                    # cannot affect these suppliers.
                    self.sharedComm.Barrier()

                    if self.sharedRank == 0:
                        Firm.sharedOutputWarehouseSnapshot[:] = (
                            Firm.sharedOutputWarehouse
                        )

                    self.sharedComm.Barrier()

                fractional_attempt = (
                    attempt_index >= full_attempts
                )

                # Countries are distributed among ranks for computation only.
                # Each coordinator writes disjoint buyer slots.
                for country_position, country_value in enumerate(
                    country_order
                ):

                    if country_position % rankNum != rank:
                        continue

                    aCountry = int(country_value)
                    all_buyer_addresses = (
                        self.globalBuyerAddressesByCountry[aCountry]
                    )

                    if all_buyer_addresses.size == 0:
                        continue

                    # Clear the whole country slice so unselected Firms in the
                    # fractional round can never inherit a previous match.
                    Firm.sharedMatchedSupplierAddress[
                        all_buyer_addresses
                    ] = -1

                    if fractional_attempt:
                        buyer_addresses = fractionalCoordinatorBuyers[aCountry]
                    else:
                        buyer_addresses = all_buyer_addresses

                    if buyer_addresses.size == 0:
                        continue

                    if not sector_suppliers:
                        continue

                    # Seed is a pure function of the economic matching problem,
                    # never of the MPI owner or coordinator rank.
                    match_rng = np.random.default_rng(
                        np.random.SeedSequence([
                            global_matching_seed,
                            period_index + 1,
                            attempt_index,
                            aSector,
                            aCountry,
                            9001,
                        ])
                    )

                    Firm.sharedMatchedSupplierAddress[
                        buyer_addresses
                    ] = match_global_buyer_addresses_to_suppliers(
                        buyer_addresses=buyer_addresses,
                        suppliers=sector_suppliers,
                        buyer_country=aCountry,
                        mult=local_preference,
                        rng=match_rng,
                        supplier_output_snapshot=(
                            Firm.sharedOutputWarehouseSnapshot
                        ),
                        buyer_refill_snapshot=(
                            Firm.sharedBuyerRefill
                        ),
                        point_mode="systematic",
                    )

                # No buyer may purchase before every country coordinator has
                # completed the global matching for this sector and round.
                self.sharedComm.Barrier()

                # Each rank executes only its locally owned buyers. Residual
                # refill stays in the shared N-vector for the next round.
                for country_value in country_order:

                    aCountry = int(country_value)

                    for aBuyer in self.firmListByCountry[aCountry]:

                        if (
                            fractional_attempt
                            and not fractionalEligibleLocal[aBuyer.uid[0]]
                        ):
                            continue

                        refill = float(
                            Firm.sharedBuyerRefill[aBuyer.address]
                        )

                        if refill <= 0.0:
                            continue

                        supplier_address = int(
                            Firm.sharedMatchedSupplierAddress[
                                aBuyer.address
                            ]
                        )

                        if supplier_address < 0:
                            continue

                        supplier = self.supplierByAddress[
                            supplier_address
                        ]

                        is_local = (
                            supplier.country
                            == aCountry
                        )

                        countTotalRefillOperations += 1

                        if is_local:
                            countTotalLocalRefillOperations += 1

                        purchased, available, remaining = self.purchaseFromSupplier(
                            supplier,
                            refill,
                        )

                        if purchased > 0.0:
                            countSuccessfulRefillOperations += 1

                            if is_local:
                                countSuccessfulLocalRefillOperations += 1

                            aBuyer.intermediateWarehouse[aSector] += purchased
                            refill -= purchased

                        Firm.sharedBuyerRefill[
                            aBuyer.address
                        ] = max(0.0, refill)

            # Final synchronization for this sector: all ranks must finish the
            # last EU27 purchase round before any owner turns residual demand
            # into imports and the shared refill vector is reused for a new sector.
            self.sharedComm.Barrier()

            for aFirm in self.localFirms:
                refill = float(
                    Firm.sharedBuyerRefill[aFirm.address]
                )

                if (
                    refill > 0.0
                    and import_enabled
                ):
                    aFirm.intermediateWarehouse[aSector] += refill
                    imports[aSector] += refill
                    self.intermediateTotalImportsByCountryAndSector[
                        aSector
                    ][aFirm.country] += refill

        # Use the same global Two-Tape, repeated rounds and local preference
        # for replacement investment goods supplied by productionType == "I".
        investmentImports = self.selectInvestmentSuppliers(
            local_preference=local_preference,
            total_attempts=total_attempts,
            full_attempts=full_attempts,
            fractionalEligibleLocal=fractionalEligibleLocal,
            fractionalCoordinatorBuyers=fractionalCoordinatorBuyers,
            country_order=country_order,
            global_matching_seed=global_matching_seed,
            period_index=period_index,
            import_enabled=import_enabled,
        )

        failedRefillOperations = (
            countTotalRefillOperations
            - countSuccessfulRefillOperations
        )

        failedLocalRefillOperations = (
            countTotalLocalRefillOperations
            - countSuccessfulLocalRefillOperations
        )

        if not params["stop_technical_print"]: print(
            "\n\ncountTotalRefillOperations",
            countTotalRefillOperations,
            "countTotalLocalRefillOperations",
            countTotalLocalRefillOperations,
            "\ncountSuccessfulRefillOperations",
            countSuccessfulRefillOperations,
            "countSuccessfulLocalRefillOperations",
            countSuccessfulLocalRefillOperations,
            "\ncountFailedRefillOperations",
            failedRefillOperations,
            "countFailedLocalRefillOperations",
            failedLocalRefillOperations,
            flush=True,
        )

        # Print residual demand and supply before moving output to the
        # potential extra-EU export warehouses.
        if not params["stop_technical_print"]:
            print("\nAfter refill")
            printDemandOfferRefill(outAddedV=False, intermediateInputs=self.intermediateInputs)

            print(
                "\nImports",
                [
                    (
                        sector + 1,
                        round(
                            value / 1_000_000,
                            2,
                        ),
                    )
                    for sector, value
                    in enumerate(imports)
                ],
            )

            print(
                "\nTotal imports",
                round(
                    sum(imports) / 1_000_000,
                    2,
                ),
            )

        supplierDiagPurchaseWorkWall = (
            time.perf_counter()
            - supplierDiagWorkWallStart
        )

        supplierDiagPurchaseWorkCpu = (
            time.process_time()
            - supplierDiagWorkCpuStart
        )

        supplierDiagWallStart = time.perf_counter()
        supplierDiagCpuStart = time.process_time()

        # All cross-rank purchases must finish before owners move residual
        # output to the potential extra-EU export warehouse.
        self.sharedComm.Barrier()

        supplierDiagPreExportBarrierWall = (
            time.perf_counter()
            - supplierDiagWallStart
        )

        supplierDiagPreExportBarrierCpu = (
            time.process_time()
            - supplierDiagCpuStart
        )

        supplierDiagExportWallStart = time.perf_counter()
        supplierDiagExportCpuStart = time.process_time()

        potentialExtraEU27Export = 0.0
        potentialInvestmentExtraEU27Export = 0.0
        self.investmentPotentialExportsArray.fill(0.0)

        # Only locally owned Firms perform this operation. Intermediate and
        # investment suppliers have separate potential-export variables.
        for supplier in self.localFirms:

            if supplier.productionType not in ("Int", "I"):
                continue

            residual_output = max(0.0, supplier.outputWarehouse)

            if supplier.productionType == "Int":
                supplier.warehouseForPotentialExtraEU27export = residual_output
                potentialExtraEU27Export += residual_output
            else:
                supplier.warehouseForPotentialInvestmentExtraEU27export = residual_output
                potentialInvestmentExtraEU27Export += residual_output

            supplier.outputWarehouse = 0.0

        # Matrices of potential firm exports by sector and by country.
        for aFirm in self.localFirms:
            self.intermediateTotalPotentialExportsByCountryAndSector[aFirm.sector][aFirm.country] +=\
                                                            aFirm.warehouseForPotentialExtraEU27export
            self.investmentTotalPotentialExportsByCountryAndSector[aFirm.sector][aFirm.country] +=\
                                                aFirm.warehouseForPotentialInvestmentExtraEU27export

        supplierDiagExportWorkWall = (
            time.perf_counter()
            - supplierDiagExportWallStart
        )

        supplierDiagExportWorkCpu = (
            time.process_time()
            - supplierDiagExportCpuStart
        )

        supplierDiagWallStart = time.perf_counter()
        supplierDiagCpuStart = time.process_time()

        self.sharedComm.Barrier()

        supplierDiagFinalBarrierWall = (
            time.perf_counter()
            - supplierDiagWallStart
        )

        supplierDiagFinalBarrierCpu = (
            time.process_time()
            - supplierDiagCpuStart
        )

        supplierDiagTotalWall = (
            time.perf_counter()
            - supplierDiagTotalWallStart
        )

        supplierDiagTotalCpu = (
            time.process_time()
            - supplierDiagTotalCpuStart
        )

        if self.superTechnicalPrint:
            print(
                f"SUPPLIER DIAG rank {rank:3d} month {self.count_months + 1:3d} | "
                f"wall: start_barrier {supplierDiagStartBarrierWall:.6f}  "
                f"purchase_work {supplierDiagPurchaseWorkWall:.6f}  "
                f"pre_export_barrier {supplierDiagPreExportBarrierWall:.6f}  "
                f"export_work {supplierDiagExportWorkWall:.6f}  "
                f"final_barrier {supplierDiagFinalBarrierWall:.6f}  "
                f"total {supplierDiagTotalWall:.6f} | "
                f"cpu: start_barrier {supplierDiagStartBarrierCpu:.6f}  "
                f"purchase_work {supplierDiagPurchaseWorkCpu:.6f}  "
                f"pre_export_barrier {supplierDiagPreExportBarrierCpu:.6f}  "
                f"export_work {supplierDiagExportWorkCpu:.6f}  "
                f"final_barrier {supplierDiagFinalBarrierCpu:.6f}  "
                f"total {supplierDiagTotalCpu:.6f}",
                flush=True,
            )

        # Current-period potential extra-EU export flows.
        if not params["stop_technical_print"]:
            print(
                "\nPotential ExtraEU27 export",
                round(
                    potentialExtraEU27Export
                    / 1_000_000,
                    2,
                ),
            )
            print(
                "\nPotential investment-goods ExtraEU27 export",
                round(
                    potentialInvestmentExtraEU27Export
                    / 1_000_000,
                    2,
                ),
            )


    def adjustCapital(self):
        """Apply monthly capital substitution and add delivered investment goods."""

        for aFirm in self.localFirms:
            capitalBefore = float(aFirm.capital)

            if aFirm.sector in (44, 45):
                # Sector 44 is imputed rents. Sector 45 keeps the substitutionRateL
                # rule already present in demand(), preserving the existing model.
                substitutionRate = self.substitutionRateL
            else:
                substitutionRate = self.substitutionRate

            capitalReduction = (
                capitalBefore
                * substitutionRate
                / 12
            )

            investmentPurchased = float(
                aFirm.investmentGoodsPurchased
            )

            if aFirm.sector == 44:
                # The special imputed-rents investment order was scaled down to the
                # simulated economy. Restore the represented full-sector scale before
                # adding delivered investment goods back to this virtual Firm's capital.
                investmentPurchased *= (
                    self.Eu_FirmNumber
                    / self.simulationFirmsExAnteNumber
                )

            aFirm.capital = max(
                0.0,
                capitalBefore
                - capitalReduction
                + investmentPurchased,
            )

        
    ###################################################################################################
    ################################# AGGREGATING RESULTS BY COUNTRY ##################################
    ###################################################################################################
    def aggregatingResultsByCountry(self):
        for j in range(27):
            self.countWorkers[27] += self.countWorkers[j]
            self.countEmployeeCompensations[27] += self.countEmployeeCompensations[j] 
            self.countCapitalR[27] += self.countCapitalR[j] 
            self.countActualAddedValue[27] += self.countActualAddedValue[j]
            self.countAFF_Firms[27] += self.countAFF_Firms[j]
            self.countAFF_WorkerNumber[27] += self.countAFF_WorkerNumber[j] 
            self.intermediateTotalSupplyByCountry[27] += self.intermediateTotalSupplyByCountry[j]
            self.intermediateTotalDemandByCountry[27] += self.intermediateTotalDemandByCountry[j]
            self.firmSubstitutions[27] += self.firmSubstitutions[j] 

            self.allfirmSubstitutionsByVendorSector[:, 27] += self.allfirmSubstitutionsByVendorSector[:,j]
            self.allfirmSubstitutionsByBuyerSector[:, 27] += self.allfirmSubstitutionsByBuyerSector[:,j]

            self.intermediateTotalImportsByCountryAndSector[:, 27] += self.intermediateTotalImportsByCountryAndSector[:,j]
            self.intermediateTotalPotentialExportsByCountryAndSector [:, 27] += self.intermediateTotalPotentialExportsByCountryAndSector[:,j]
            self.investmentTotalImportsByCountryAndSector[:, 27] += self.investmentTotalImportsByCountryAndSector[:,j]
            self.investmentTotalPotentialExportsByCountryAndSector[:, 27] += self.investmentTotalPotentialExportsByCountryAndSector[:,j]

            # remember that you have to add the aggregation by sector!!!!!!!!!!!!!!!!!!!!!!


    # aggregate results over ranks
    def aggregatingRanks(self):
        if rankNum == 1: return

        # The number of firms is constant and has to be reduced only once.
        if self.count_months == 0:
            totalFirmNumber = comm.reduce(self.countActualExPostFirmNumber, op=MPI.SUM, root=0)
            if rank == 0:
                self.countActualExPostFirmNumber = totalFirmNumber

        # Pack all monthly results in one contiguous NumPy array.
        # In this way only one MPI.Reduce is required each month.
        if self.detailedTiming:
            packWallStart = time.perf_counter()
            packCpuStart = time.process_time()
        localResults = np.concatenate((
            np.asarray(self.countWorkers, dtype=np.float64),
            np.asarray(self.countEmployeeCompensations, dtype=np.float64),
            np.asarray(self.countCapitalR, dtype=np.float64),
            np.asarray(self.countAFF_Firms, dtype=np.float64),
            np.asarray(self.countAFF_WorkerNumber, dtype=np.float64),
            np.asarray(self.intermediateTotalSupplyBySector, dtype=np.float64),
            np.asarray(self.intermediateTotalSupplyByCountry, dtype=np.float64),
            np.asarray(self.intermediateTotalDemandBySector, dtype=np.float64),
            np.asarray(self.intermediateTotalDemandByCountry, dtype=np.float64),
            np.asarray(self.firmSubstitutions, dtype=np.float64),
            np.asarray(self.allfirmSubstitutionsByVendorSector, dtype=np.float64).ravel(),
            np.asarray(self.allfirmSubstitutionsByBuyerSector, dtype=np.float64).ravel(),
            np.asarray(self.intermediateTotalSupplyBySectorAndCountry, dtype=np.float64).ravel(),
            np.asarray(self.intermediateTotalDemandBySectorAndCountry, dtype=np.float64).ravel(),
            np.asarray(self.countActualAddedValue, dtype=np.float64),
            np.asarray(self.intermediateTotalImportsByCountryAndSector, dtype=np.float64).ravel(),
            np.asarray(self.intermediateTotalPotentialExportsByCountryAndSector, dtype=np.float64).ravel(),
            np.asarray(self.investmentTotalImportsByCountryAndSector, dtype=np.float64).ravel(),
            np.asarray(self.investmentTotalPotentialExportsByCountryAndSector, dtype=np.float64).ravel()
        ))
        
        if self.detailedTiming:
            self.reduceTimingWall["pack"] = time.perf_counter() - packWallStart
            self.reduceTimingCpu["pack"] = time.process_time() - packCpuStart

        if rank == 0:
            totalResults = np.empty_like(localResults)
        else:
            totalResults = None

        if self.detailedTiming:
            reduceWallStart = time.perf_counter()
            reduceCpuStart = time.process_time()
        comm.Reduce(localResults, totalResults, op=MPI.SUM, root=0)
        if self.detailedTiming:
            self.reduceTimingWall["reduce"] = time.perf_counter() - reduceWallStart
            self.reduceTimingCpu["reduce"] = time.process_time() - reduceCpuStart

        if rank != 0: return

        # Unpack the reduced results on rank 0.
        if self.detailedTiming:
            unpackWallStart = time.perf_counter()
            unpackCpuStart = time.process_time()
        p = 0

        self.countWorkers = totalResults[p:p+28].astype(np.int64).tolist()
        p += 28
        self.countEmployeeCompensations = totalResults[p:p+28].tolist()
        p += 28
        self.countCapitalR = totalResults[p:p+28].tolist()
        p += 28
        self.countAFF_Firms = totalResults[p:p+28].astype(np.int64).tolist()
        p += 28
        self.countAFF_WorkerNumber = totalResults[p:p+28].astype(np.int64).tolist()
        p += 28

        self.intermediateTotalSupplyBySector = totalResults[p:p+64].tolist()
        p += 64
        self.intermediateTotalSupplyByCountry = totalResults[p:p+28].tolist()
        p += 28
        self.intermediateTotalDemandBySector = totalResults[p:p+64].tolist()
        p += 64
        self.intermediateTotalDemandByCountry = totalResults[p:p+28].tolist()
        p += 28
        self.firmSubstitutions = totalResults[p:p+28].tolist()
        p += 28

        self.allfirmSubstitutionsByVendorSector = totalResults[p:p+65*28].reshape(65,28)
        p += 65*28
        self.allfirmSubstitutionsByBuyerSector = totalResults[p:p+65*28].reshape(65,28)
        p += 65*28

        self.intermediateTotalSupplyBySectorAndCountry = totalResults[p:p+64*28].reshape(64,28).tolist()
        p += 64*28
        self.intermediateTotalDemandBySectorAndCountry = totalResults[p:p+64*28].reshape(64,28).tolist()
        p += 64*28

        self.countActualAddedValue = totalResults[p:p+28].tolist()
        p += 28 
        
        self.intermediateTotalImportsByCountryAndSector =  totalResults[p:p+65*28].reshape(65,28)
        p += 65*28
        self.intermediateTotalPotentialExportsByCountryAndSector =  totalResults[p:p+65*28].reshape(65,28)
        p += 65*28
        self.investmentTotalImportsByCountryAndSector = totalResults[p:p+65*28].reshape(65,28)
        p += 65*28
        self.investmentTotalPotentialExportsByCountryAndSector = totalResults[p:p+65*28].reshape(65,28)
        p += 65*28 # here to ease the next entry of the big results matrix

        
        if self.detailedTiming:
            self.reduceTimingWall["unpack"] = time.perf_counter() - unpackWallStart
            self.reduceTimingCpu["unpack"] = time.process_time() - unpackCpuStart

    # show results
    def showResults(self):
        
        if rank != 0: return

        self.propFactor = self.Eu_FirmNumber / self.countActualExPostFirmNumber # redefined here to be updated with rankNum > 1

        print("\n", "MONTH",  (self.count_months % 12) + 1 , "YEAR", (self.count_months // 12) + 1)
        print("\nLocal preference", params['localPreference'], "max_refill_repeat", params['max_refill_repeat'])

        
        generateOutput("totEU27", sys.stdout, self.AFF_FirmNumber, self.countAFF_Firms[27], self.propFactor, self.AFF_WorkerNumber,\
                    self.countAFF_WorkerNumber[27], self.Eu_FirmNumber, self.countActualExPostFirmNumber, self.Eu_WorkerNumber,\
                    self.countWorkers[27], self.Eu_EmployeeCompensations, self.countEmployeeCompensations[27], self.Eu_GDP_withVAT2022,\
                    self.Eu_AddedValue2022, self.countCapitalR[27], self.countActualAddedValue[27], self.markupTentative, self.Eu_Intermediate2022,\
                    self.intermediateTotalDemandByCountry[27], self.substitutionRate, self.substitutionRateL, self.firmSubstitutions[27],\
                    self.allfirmSubstitutionsByBuyerSector[:,27], self.simulationFirmsExAnteNumber, self.allfirmSubstitutionsByVendorSector[:,27],\
                    self.intermediateTotalImportsByCountryAndSector[:,27], self.intermediateTotalPotentialExportsByCountryAndSector[:,27],\
                    self.investmentTotalImportsByCountryAndSector[:,27], self.investmentTotalPotentialExportsByCountryAndSector[:,27])

        if self.selected_countries == []: return

        for aCountry in self.selected_countries:
            if os.path.exists(self.eu_countries[aCountry]+".txt") and self.count_months == 1: os.remove(self.eu_countries[aCountry]+".txt")
            
            with open(self.eu_countries[aCountry]+'.txt', 'a', newline='') as txtfile:              
                #print(txtfile)
                print("MONTH", (self.count_months % 12) + 1, file = txtfile)
                generateOutput(self.eu_countries[aCountry], txtfile, self.AFF_FirmNumber,
                                self.countAFF_Firms[aCountry], self.propFactor,
                                self.AFF_WorkerNumber, self.countAFF_WorkerNumber[aCountry],
                                self.Eu_FirmNumber, self.countActualExPostFirmNumber,
                                self.Eu_WorkerNumber, self.countWorkers[aCountry],
                                self.Eu_EmployeeCompensations, self.countEmployeeCompensations[aCountry],
                                self.Eu_GDP_withVAT2022, self.Eu_AddedValue2022,
                                self.countCapitalR[aCountry], 
                                self.countActualAddedValue[aCountry],
                                self.markupTentative,
                                self.Eu_Intermediate2022, self.intermediateTotalDemandByCountry[aCountry],
                                self.substitutionRate, self.substitutionRateL,
                                self.firmSubstitutions[aCountry],
                                self.allfirmSubstitutionsByBuyerSector[:, aCountry],
                                self.simulationFirmsExAnteNumber,
                                self.allfirmSubstitutionsByVendorSector[:, aCountry],
                                self.intermediateTotalImportsByCountryAndSector[:, aCountry],
                                self.intermediateTotalPotentialExportsByCountryAndSector[:, aCountry],
                                self.investmentTotalImportsByCountryAndSector[:, aCountry],
                                self.investmentTotalPotentialExportsByCountryAndSector[:, aCountry]
                              )

        
        print("\n", 'Add. Val. basic p. by country', "\n")
        for aCountry in range(len(self.eu_countries)):
            generateGDPOutput(self.eu_countries[aCountry], sys.stdout, self.propFactor, self.countActualAddedValue[aCountry])
                        #self.countEmployeeCompensations[aCountry], self.countCapitalR[aCountry], self.markupTentative)

    
    #finish
    def finish(self):
        print("\n\n")
        print(f"Rank {rank}, global time {T()}, cpu time {Tc()}")
        print("\nRank ", rank, "concluded")
    
    def start(self):
        runner.execute()

def run():

    model = Model() 
    model.start()
    model.sharedComm.Barrier()
    model.supplierLockWin.Free()
    model.productionTypeCodeWin.Free()
    model.sectorWin.Free()
    model.countryWin.Free()
    model.matchedSupplierAddressWin.Free()
    model.buyerRefillWin.Free()
    model.targetAddedValueWin.Free()
    model.lostSalesWin.Free()
    model.outputWarehouseSnapshotWin.Free()
    model.outputWarehouseWin.Free()
    model.sharedComm.Free()
    return model # to access model when the execution is finished
    
model=run() #run() returns model, useful to access all the variables of the model

#file name: ff_with_class_limits.csv

rank 0 tableOfInvestmentShare 2
rank 0 how many firms 1000000
rank 0 use smart capital 1
1000010 firms created in rank  0
propFactor 42.25145148548515 in rank 0
FirmProxy remote suppliers per rank min 0, max 0, shallow object size 0 bytes
Rank 0 Starting phase concluded: global time 81.02498292922974, cpu time 78.94702199999999

 MONTH 1 YEAR 1

Local preference 20 max_refill_repeat 3.8
INT IMPORT 4222.48
[(0, 400.35), (1, 31.11), (2, 6.55), (3, 437.37), (4, 116.1), (5, 79.23), (6, 2.06), (7, 0.0), (8, 11.11), (9, 3.38), (10, 46.04), (11, 0.0), (12, 23.09), (13, 24.95), (14, 56.66), (15, 26.81), (16, 5.63), (17, 5.42), (18, 8.03), (19, 35.01), (20, 0.0), (21, 42.5), (22, 74.58), (23, 52.27), (24, 12.07), (25, 127.39), (26, 212.07), (27, 50.67), (28, 381.22), (29, 68.41), (30, 185.54), (31, 0.0), (32, 0.14), (33, 144.07), (34, 14.15), (35, 76.04), (36, 35.39), (37, 1.21), (38, 44.33), (39, 84.22), (40, 40.21), (41, 0.98), (42, 181.04), (43, 0.0), (44, 103.97), (45, 346.34), (46, 79.1), 

---------  
   
Italy simulated GDP per month: 133108  
Italy GDP in 2022: 1946479 --> 1946479 / (12 * 1.22) = 132956.2 ca. to get monthly at basic price GDP 

**to run the code in a multi-rank way, from the terminal launch:**  

mpirun -n x ipython modelEU27.ipynb

**using Ryzen Threadripper:**

mpirun -n X -bind-to core ipython modelEU27.ipynb

*best choice: X = 40*

**to access the contents of the program after the execution**

%whos - gives the list of the variables  
context - is the address of the context; e.g., context.agent((0,0,0)).outputWarehouse  
model - is the address of the model, returned by run(); e.g., model.eu_countries

In [16]:
for i in range(len(model.investmentTotalSupplyBySector)):
   print(i, round( model.investmentTotalSupplyBySector[i]/1000, 2))

0 12358.4
1 865.31
2 79.2
3 164.96
4 0.0
5 430.51
6 7701.99
7 0.0
8 0.0
9 0.0
10 0.0
11 0.0
12 7220.14
13 9424.15
14 1707.38
15 122285.09
16 257765.59
17 69217.83
18 768702.79
19 424621.1
20 203421.55
21 97492.79
22 121262.52
23 57.32
24 0.0
25 0.0
26 2548374.13
27 45394.53
28 230648.53
29 68598.59
30 34983.83
31 2123.35
32 1363.68
33 164.84
34 0.0
35 680.34
36 55660.58
37 27758.19
38 0.0
39 638029.55
40 193.59
41 3244.9
42 1331.34
43 0.0
44 76675.61
45 83718.49
46 136671.45
47 367255.53
48 0.0
49 53.35
50 588.71
51 33.33
52 40.65
53 24061.73
54 3283.32
55 110.1
56 0.0
57 0.0
58 8408.58
59 12.57
60 0.0
61 1655.04
62 0.0
63 0.0


In [17]:
#3_258_812_096.8699446
#3_320_258.70

sum(model.investmentTotalSupplyBySector)/1000 *12 *42



3258812096.8699446

In [13]:
sum(model.investmentTotalSupplyBySector)/1000000*12 *42
3_258_812.09686994

3258812.09686994

In [14]:
for i in range(65):
    print(i, round(model.allfirmSubstitutionsByBuyerSector[i,27]/1000, 2))


np.sum(model.allfirmSubstitutionsByBuyerSector[:, 27]) / 1000 *12 *42

0 171.64
1 20.59
2 6.57
3 52.82
4 78.92
5 11.12
6 10.14
7 13.06
8 4.96
9 19.81
10 122.8
11 125.57
12 30.36
13 21.77
14 23.98
15 68.51
16 49.17
17 31.96
18 82.21
19 81.71
20 27.39
21 19.34
22 19.02
23 293.39
24 16.11
25 56.66
26 129.01
27 76.67
28 302.68
29 220.31
30 86.12
31 22.32
32 6.86
33 85.42
34 18.29
35 165.86
36 26.42
37 17.51
38 34.34
39 134.83
40 49.96
41 13.61
42 12.33
43 986.35
44 1260.43
45 36.93
46 14.22
47 10.09
48 4.09
49 5.32
50 7.52
51 8.74
52 0.89
53 15.48
54 216.81
55 59.82
56 210.17
57 107.82
58 8.11
59 5.99
60 0.37
61 0.05
62 0.37
63 0.0
64 0.0


np.float64(2934123.668912314)

In [15]:
2_934_123

2934123

In [20]:
t=0
for aFirm in context.agents(agent_type=0):
    t+=aFirm.investmentGoodsPurchased

In [24]:
t*12*42.25

2951581659.822585

In [32]:
t2=0
for aFirm in context.agents(agent_type=0):
    if aFirm.productionType =="I": t2+=(aFirm.outputWarehouse+aFirm.intermediateWarehouseWithdraw)

t2

3941373182.1666846

In [ ]:
3_941_373_182.1666846

In [31]:
(1407.65+3_941.37)*12*42.25

2711953.14